In [ ]:
# ---- Configuration: edit config/config.yaml (see README) ----
# PyDESeq2 analysis with PCA-weighted re-analysis, Venn diagrams, MA/volcano plots,
# Enrichr and GSEA (gseapy). Input: <bulk_rna_workdir>/gene_counts.csv written by
# alignment/03_count_reads.sh (columns: gene_id, gene_name, one column per sample).
# Outputs are written to <pydeseq2_workdir>.
import os, sys
from pathlib import Path
_root = Path(os.environ.get("PROJECT_ROOT") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "config.py").exists()))
os.environ["PROJECT_ROOT"] = str(_root)
sys.path.insert(0, str(_root))
from utils.config import cfg_path
COUNTS_CSV = os.path.join(cfg_path("bulk_rna_workdir"), "gene_counts.csv")
os.chdir(cfg_path("pydeseq2_workdir", create=True))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
data = pd.read_csv(COUNTS_CSV, index_col=0).drop(columns="gene_name")   # gene_id x samples
data= data[data.sum(axis = 1) > 0]
data.head(1000000)

In [ ]:
data1=data.T
data1.head(1000000)

In [ ]:
metadata = pd.DataFrame(zip(data1.index, ['Control-Vehicle','Control-Vehicle','Control-Vehicle','Control-Treated','Control-Treated','Control-Treated','MutA-Vehicle', 'MutA-Vehicle', 'MutA-Vehicle','MutA-Treated', 'MutA-Treated', 'MutA-Treated', 'MutB-Vehicle', 'MutB-Vehicle', 'MutB-Vehicle','MutB-Treated', 'MutB-Treated', 'MutB-Treated']),
                        columns = ['Sample', 'Condition'])
metadata = metadata.set_index('Sample')
metadata

In [ ]:
dds = DeseqDataSet(counts=data1,
            metadata=metadata,
            design_factors=["Condition"])

In [ ]:
dds.deseq2()

In [ ]:
dds

In [ ]:
# Assuming you have already performed the DESeq2 analysis and stored the results in a DESeq2 object called "dds"

# Extract dispersion values
dispersions = dds.varm['dispersions']

# Extract mean expression levels
mean_expression = dds.varm['_normed_means']

# Plot dispersion estimates
plt.scatter(np.log2(mean_expression), np.log2(dispersions), color='blue', alpha=0.5)
plt.xlabel('Log2 Mean Expression')
plt.ylabel('Log2 Dispersion')
plt.title('Dispersion Estimates')

plt.savefig('Dispersion Plot.jpg', bbox_inches="tight")

plt.show()

In [ ]:
Normalised_Counts=dds.layers['normed_counts']
Norm_Counts=pd.DataFrame(Normalised_Counts)
Norm_Counts.head(2000000)

In [ ]:
Norm_Counts.set_index(data1.index, inplace=True)
Norm_Counts

In [ ]:
Norm_Counts=Norm_Counts.T
Norm_Counts

In [ ]:
Norm_Counts.set_index(data.index, inplace=True)
Normalised_dataset=Norm_Counts
Normalised_dataset.head(2000000)

In [ ]:
SqrtVariance = Normalised_dataset.copy()
SqrtVariance= SqrtVariance[SqrtVariance.sum(axis = 1) > 0]
SqrtVariance.head(2000000)

In [ ]:
import numpy as np

# Assuming Normalised_dataset is your DataFrame
SqrtVariance['variance'] = SqrtVariance.var(axis=1)
SqrtVariance['sqrt_variance'] = np.sqrt(SqrtVariance['variance'])

# Viewing the first 2000000 rows
SqrtVariance.head(2000000)


In [ ]:
# Assuming you have already performed the DESeq2 analysis and stored the results in a DESeq2 object called "dds"

# Extract dispersion values
sqrt_var = SqrtVariance['sqrt_variance']

# Extract mean expression levels
mean_expression = dds.varm['_normed_means']

# Plot dispersion estimates
plt.scatter(mean_expression, sqrt_var, color='blue', alpha=0.5)
plt.xlabel('Mean Expression')
plt.ylabel('Sqaure Root of Variance')
plt.title('Sqaure Root of Variance Analysis')

plt.savefig('Sq root of variance Scatter.jpg', bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Sort the data by mean expression for better visualization
sorted_indices = np.argsort(mean_expression)
sorted_mean_expression = mean_expression[sorted_indices]
sorted_sqrt_var = sqrt_var[sorted_indices]

plt.plot(sorted_mean_expression, sorted_sqrt_var, color='blue')
plt.xlabel('Mean Expression')
plt.ylabel('Sqaure Root of Variance')
plt.title('Sqaure Root of Variance Analysis')

plt.savefig('Sq root of variance.jpg', bbox_inches="tight")

plt.show()


In [ ]:
Normalised_dataset.head(2000000)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(Normalised_dataset.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(15, 8))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(Normalised_dataset.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(Normalised_dataset.columns) + 1))
ax.grid(True)

plt.savefig('PCA Components .jpg', bbox_inches="tight")

plt.show()

# Step 4: Eigenvalues Extraction
eigenvalues = pca_all.explained_variance_ratio_

# Step 5: Scree Plot
plt.figure(figsize=(15, 8))
plt.plot(np.arange(1, len(eigenvalues) + 1), eigenvalues, marker='o', linestyle='-')
plt.title('Scree Plot')
plt.xlabel('Number of Components')
plt.ylabel('Eigenvalue')
plt.xticks(np.arange(1, len(Normalised_dataset.columns) + 1))  # Corrected line
plt.grid(True)

plt.savefig('PCA Scree Plot .jpg', bbox_inches="tight")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming eigenvalues are already computed
eigenvalues = pca_all.explained_variance_ratio_

# Step 1: Calculate cumulative sum of eigenvalues
cumulative_eigenvalues = np.cumsum(eigenvalues)

# Step 2: Calculate expected values for each component
total_variance = np.sum(eigenvalues)
expected_values = np.arange(1, len(eigenvalues) + 1) * total_variance / len(eigenvalues)

# Step 3: Calculate Chi-square values
chi_square_values = np.square(cumulative_eigenvalues - expected_values) / expected_values
print(chi_square_values)

# Step 4: Plot Chi-square values
plt.figure(figsize=(15, 8))
plt.plot(np.arange(1, len(chi_square_values) + 1), chi_square_values, marker='o', linestyle='-')
plt.title('Chi-square Analysis for Scree Plot')
plt.xlabel('Number of Components')
plt.ylabel('Chi-square Value')
plt.xticks(np.arange(1, len(chi_square_values) + 1))
plt.grid(True)

plt.savefig('PCA Chi-square Analysis.jpg', bbox_inches="tight")

plt.show()


In [ ]:

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PC1 vs PC2.jpg', bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T



# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_2.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_2.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PC1 vs PC3.jpg', bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T



# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 1], y=pca_2_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 1], y=pca_2_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y = pca_2_result[i, 1], pca_2_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_2.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_2.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PC2 vs PC3.jpg', bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T



# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 2], y=pca_2_result[i, 3], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 2], y=pca_2_result[i, 3], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y = pca_2_result[i, 2], pca_2_result[i, 3]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 3 ({pca_2.explained_variance_ratio_[2]*100:.2f}%)')
plt.ylabel(f'Principal Component 4 ({pca_2.explained_variance_ratio_[3]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC3: {pca_2.explained_variance_ratio_[2]*100:.2f}%")
print(f"Explained Variance Ratio PC4: {pca_2.explained_variance_ratio_[3]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PC3 vs PC4.jpg', bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T



# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 3], y=pca_2_result[i, 4], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 3], y=pca_2_result[i, 4], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y = pca_2_result[i, 3], pca_2_result[i, 4]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 4 ({pca_2.explained_variance_ratio_[3]*100:.2f}%)')
plt.ylabel(f'Principal Component 5 ({pca_2.explained_variance_ratio_[4]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC4: {pca_2.explained_variance_ratio_[3]*100:.2f}%")
print(f"Explained Variance Ratio PC5: {pca_2.explained_variance_ratio_[4]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PC4 vs PC5.jpg', bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T



# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 4], y=pca_2_result[i, 5], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 4], y=pca_2_result[i, 5], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y = pca_2_result[i, 4], pca_2_result[i, 5]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 5 ({pca_2.explained_variance_ratio_[4]*100:.2f}%)')
plt.ylabel(f'Principal Component 6 ({pca_2.explained_variance_ratio_[5]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC5: {pca_2.explained_variance_ratio_[4]*100:.2f}%")
print(f"Explained Variance Ratio PC6: {pca_2.explained_variance_ratio_[5]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PC5 vs PC6.jpg', bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Log transformation
#log_transformed_data2 = np.log1p(data1)

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized2 = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed2 = data_standardized2.T

# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment for 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_2_result[i, 0], pca_2_result[i, 1], pca_2_result[i, 2],  color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_2_result[i, 0], pca_2_result[i, 1], pca_2_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y, z = pca_2_result[i, 0],  pca_2_result[i, 1], pca_2_result[i, 2]
    ax.text(x, y, z, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1, PC2, and PC3 values
ax.set_xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_zlabel(f'Principal Component 3 ({pca_2.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis (3D)', fontweight='bold')

# Add grids
ax.grid(True)

# Customized legend box
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PCA PC1 vs PC2 vs PC3.jpg', bbox_inches="tight")

# Show the 3D plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Log transformation
#log_transformed_data2 = np.log1p(data1)

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized2 = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed2 = data_standardized2.T

# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment for 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_2_result[i, 2], pca_2_result[i, 3], pca_2_result[i, 4],  color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_2_result[i, 2], pca_2_result[i, 3], pca_2_result[i, 4], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y, z = pca_2_result[i, 2],  pca_2_result[i, 3], pca_2_result[i, 4]
    ax.text(x, y, z, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1, PC2, and PC3 values
ax.set_xlabel(f'Principal Component 3 ({pca_2.explained_variance_ratio_[2]*100:.2f}%)')
ax.set_ylabel(f'Principal Component 4 ({pca_2.explained_variance_ratio_[3]*100:.2f}%)')
ax.set_zlabel(f'Principal Component 5 ({pca_2.explained_variance_ratio_[4]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis (3D)', fontweight='bold')

# Add grids
ax.grid(True)

# Customized legend box
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PCA PC3 vs PC4 vs PC5.jpg', bbox_inches="tight")

# Show the 3D plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Log transformation
#log_transformed_data2 = np.log1p(data1)

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized2 = std_scaler.fit_transform(Normalised_dataset)

# Transpose the data for PCA along columns
data_transposed2 = data_standardized2.T

# Manually assign colors for each column
column_colors = {
    'OrgB_Control_Vehicle_n1': 'lightgreen',
    'OrgB_Control_Vehicle_n2': 'lightgreen',
    'OrgB_Control_Vehicle_n3': 'lightgreen',
    'OrgB_Control_Treated_n1': 'green',
    'OrgB_Control_Treated_n2': 'green',
    'OrgB_Control_Treated_n3': 'green',
    'OrgB_MutA_Vehicle_n1': 'lightcoral',
    'OrgB_MutA_Vehicle_n2': 'lightcoral',
    'OrgB_MutA_Vehicle_n3': 'lightcoral',
    'OrgB_MutA_Treated_n1': 'red',
    'OrgB_MutA_Treated_n2': 'red',
    'OrgB_MutA_Treated_n3': 'red',
    'OrgB_MutB_Vehicle_n1': 'lightblue',
    'OrgB_MutB_Vehicle_n2': 'lightblue',
    'OrgB_MutB_Vehicle_n3': 'lightblue',
    'OrgB_MutB_Treated_n1': 'blue',
    'OrgB_MutB_Treated_n2': 'blue',
    'OrgB_MutB_Treated_n3': 'blue',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n2': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n1': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n1': 'MutB Treated',
}

# Custom labels for specific data points
custom_labels = {
    'OrgB_Control_Vehicle_n1': 'Control Vehicle',
    'OrgB_Control_Treated_n3': 'Control Treated',
    'OrgB_MutA_Vehicle_n1': 'MutA Vehicle',
    'OrgB_MutA_Treated_n3': 'MutA Treated',
    'OrgB_MutB_Vehicle_n1': 'MutB Vehicle',
    'OrgB_MutB_Treated_n2': 'MutB Treated'
    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=6)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.columns)

# Set up the plotting environment for 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Normalised_dataset.columns):
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_2_result[i, 3], pca_2_result[i, 4], pca_2_result[i, 5],  color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Normalised_dataset.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_2_result[i, 3], pca_2_result[i, 4], pca_2_result[i, 5], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Normalised_dataset.columns.get_loc(col_name)
    x, y, z = pca_2_result[i, 3],  pca_2_result[i, 4], pca_2_result[i, 5]
    ax.text(x, y, z, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1, PC2, and PC3 values
ax.set_xlabel(f'Principal Component 4 ({pca_2.explained_variance_ratio_[3]*100:.2f}%)')
ax.set_ylabel(f'Principal Component 5 ({pca_2.explained_variance_ratio_[4]*100:.2f}%)')
ax.set_zlabel(f'Principal Component 6 ({pca_2.explained_variance_ratio_[5]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis (3D)', fontweight='bold')

# Add grids
ax.grid(True)

# Customized legend box
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.savefig('PCA PC4 vs PC5 vs PC6.jpg', bbox_inches="tight")

# Show the 3D plot
plt.show()

In [ ]:
# Extract PC loadings
pc_loadings = pca_2.components_

# Create a DataFrame with PC loadings for all genes
pc_loadings= pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2', 'PC3', 'PC4','PC5', 'PC6'], index=Normalised_dataset.index)
pc_loadings.to_csv('pc_loadings.csv')
pc_loadings

In [ ]:
# Convert the Series serum_csf_all_pc3_sorted to a DataFrame
pc1_sorted_df = pd.DataFrame(pc_loadings, columns=['PC1'])
pc1_sorted_df

# Sort PC loadings by PC1 in descending order
pc1_sorted_df = pc1_sorted_df.sort_values(by='PC1', ascending=False)
pc1_sorted_df.to_csv('pc1_sorted_df.csv')
pc1_sorted_df

In [ ]:
# Convert the Series serum_csf_all_pc3_sorted to a DataFrame
pc2_sorted_df = pd.DataFrame(pc_loadings, columns=['PC2'])
pc2_sorted_df

# Sort PC loadings by PC3 in descending order
pc2_sorted_df = pc2_sorted_df.sort_values(by='PC2', ascending=False)
pc2_sorted_df.to_csv('pc2_sorted_df.csv')
pc2_sorted_df


In [ ]:
# Convert the Series serum_csf_all_pc3_sorted to a DataFrame
pc3_sorted_df = pd.DataFrame(pc_loadings, columns=['PC3'])
pc3_sorted_df

# Sort PC loadings by PC3 in descending order
pc3_sorted_df = pc3_sorted_df.sort_values(by='PC3', ascending=False)
pc3_sorted_df.to_csv('pc3_sorted_df.csv')
pc3_sorted_df


In [ ]:
# Convert the Series serum_csf_all_pc3_sorted to a DataFrame
pc4_sorted_df = pd.DataFrame(pc_loadings, columns=['PC4'])
pc4_sorted_df

# Sort PC loadings by PC4 in descending order
pc4_sorted_df = pc4_sorted_df.sort_values(by='PC4', ascending=False)
pc4_sorted_df.to_csv('pc4_sorted_df.csv')
pc4_sorted_df


In [ ]:
# Convert the Series serum_csf_all_pc3_sorted to a DataFrame
pc5_sorted_df = pd.DataFrame(pc_loadings, columns=['PC5'])
pc5_sorted_df

# Sort PC loadings by PC5 in descending order
pc5_sorted_df = pc5_sorted_df.sort_values(by='PC5', ascending=False)
pc5_sorted_df.to_csv('pc5_sorted_df.csv')
pc5_sorted_df


In [ ]:
# Convert the Series serum_csf_all_pc3_sorted to a DataFrame
pc6_sorted_df = pd.DataFrame(pc_loadings, columns=['PC6'])
pc6_sorted_df

# Sort PC loadings by PC6 in descending order
pc6_sorted_df = pc6_sorted_df.sort_values(by='PC6', ascending=False)
pc6_sorted_df.to_csv('pc6_sorted_df.csv')
pc6_sorted_df


In [ ]:
Normalised_dataset.head(100000)

In [ ]:
data2 = pd.read_csv(COUNTS_CSV, index_col=0)   # gene_id index + gene_name column
data2.head(1000000)

In [ ]:
stat_res1 = DeseqStats(dds, contrast = ('Condition','MutA-Vehicle','MutB-Vehicle'))
stat_res1.summary()
stat_res1=stat_res1.results_df
stat_res1=pd.merge(stat_res1, data2[['gene_name']], left_index=True, right_index=True)
stat_res1

In [ ]:
# Copy the downregulated genes data
stat_res1_rnk = stat_res1.copy()

# Calculate the rank based on adjusted p-value
stat_res1_rnk['Rank'] = stat_res1.stat

# Sort the genes by rank in descending order
stat_res1_rnk = stat_res1_rnk.sort_values('Rank', ascending=False)
stat_res1_rnk

In [ ]:
res1 = stat_res1_rnk
res1

In [ ]:
MutA_vs_MutB_vehicle_sigs = res1[(res1.padj < 0.05)]
MutA_vs_MutB_vehicle_sigs.to_csv('MutA_vs_MutB_vehicle_sigs.csv', index=True)
MutA_vs_MutB_vehicle_sigs

In [ ]:
stat_res2 = DeseqStats(dds, contrast = ('Condition','Control-Treated','Control-Vehicle'))
stat_res2.summary()
stat_res2=stat_res2.results_df
stat_res2=pd.merge(stat_res2, data2[['gene_name']], left_index=True, right_index=True)
stat_res2

In [ ]:
# Copy the downregulated genes data
stat_res2_rnk = stat_res2.copy()

# Calculate the rank based on adjusted p-value
stat_res2_rnk['Rank'] = stat_res2.stat

# Sort the genes by rank in descending order
stat_res2_rnk = stat_res2_rnk.sort_values('Rank', ascending=False)
stat_res2_rnk

In [ ]:
res2 = stat_res2_rnk
res2

In [ ]:
Control_sigs = res2[(res2.padj < 0.05)]
Control_sigs.to_csv('Control_sigs.csv', index=True)
Control_sigs

In [ ]:
stat_res3 = DeseqStats(dds, contrast = ('Condition','MutA-Treated','MutA-Vehicle'))
stat_res3.summary()
stat_res3=stat_res3.results_df
stat_res3=pd.merge(stat_res3, data2[['gene_name']], left_index=True, right_index=True)
stat_res3

In [ ]:
# Copy the downregulated genes data
stat_res3_rnk = stat_res3.copy()

# Calculate the rank based on adjusted p-value
stat_res3_rnk['Rank'] = stat_res3.stat

# Sort the genes by rank in descending order
stat_res3_rnk = stat_res3_rnk.sort_values('Rank', ascending=False)
stat_res3_rnk

In [ ]:
res3 = stat_res3_rnk
res3

In [ ]:
MutA_sigs = res3[(res3.padj < 0.05)]
MutA_sigs.to_csv('MutA_sigs.csv', index=True)
MutA_sigs

In [ ]:
stat_res4 = DeseqStats(dds, contrast = ('Condition','MutB-Treated','MutB-Vehicle'))
stat_res4.summary()
stat_res4=stat_res4.results_df
stat_res4=pd.merge(stat_res4, data2[['gene_name']], left_index=True, right_index=True)
stat_res4

In [ ]:
# Copy the downregulated genes data
stat_res4_rnk = stat_res4.copy()

# Calculate the rank based on adjusted p-value
stat_res4_rnk['Rank'] = stat_res4.stat

# Sort the genes by rank in descending order
stat_res4_rnk = stat_res4_rnk.sort_values('Rank', ascending=False)
stat_res4_rnk

In [ ]:
res4 = stat_res4_rnk
res4

In [ ]:
MutB_sigs = res4[(res4.padj < 0.05)]
MutB_sigs.to_csv('MutB_sigs.csv', index=True)
MutB_sigs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

# Extracting all indices from each set of significant genes
gene1a_data = Control_sigs.copy()
gene1b_data = MutA_sigs.copy()
gene1c_data = MutB_sigs.copy()

# Calculate the total number of unique genes
total_genes_sigs = len(set.union(set(gene1a_data.index), set(gene1b_data.index), set(gene1c_data.index)))

# Create a Venn diagram with equal-sized circles
venn_labels_sigs = {
    '100': len(set(gene1a_data.index) - set(gene1b_data.index) - set(gene1c_data.index)),
    '010': len(set(gene1b_data.index) - set(gene1a_data.index) - set(gene1c_data.index)),
    '001': len(set(gene1c_data.index) - set(gene1a_data.index) - set(gene1b_data.index)),
    '110': len(set(gene1a_data.index) & set(gene1b_data.index) - set(gene1c_data.index)),
    '101': len(set(gene1a_data.index) & set(gene1c_data.index) - set(gene1b_data.index)),
    '011': len(set(gene1b_data.index) & set(gene1c_data.index) - set(gene1a_data.index)),
    '111': len(set(gene1a_data.index) & set(gene1b_data.index) & set(gene1c_data.index))
}

venn_labels_sigs = {key: f"{value}" for key, value in venn_labels_sigs.items()}

# Define custom colors for each set
custom_colors = {
    '100': 'green',   # Control
    '010': 'red',  # MutA
    '001': 'blue',    # MutB
}

# Create a Venn diagram
venn_sigs = venn3([set(gene1a_data.index), set(gene1b_data.index), set(gene1c_data.index)],
                  set_labels=('Control', 'MutA', 'MutB'),
                  normalize_to=total_genes_sigs)

# Set custom colors for each set
for key, value in custom_colors.items():
    venn_sigs.get_patch_by_id(key).set_color(value)

# Set labels for each region in the Venn diagram
for key, value in venn_labels_sigs.items():
    text = venn_sigs.get_label_by_id(key)
    text.set_text(value)
    text.set_fontweight('bold')  # set text to bold

# Set all text in the plot to bold
for text in venn_sigs.set_labels + venn_sigs.subset_labels:
    text.set_fontweight('bold')

# Display the plot
plt.title("Venn Diagram - Differentially Expressed Genes", y=1.08, fontweight='bold')
plt.savefig('Venn Diagram - Differentially Expressed Genes.jpg')
plt.show()

# Save genes to CSV
gene1a_data.to_csv('Genes_control_allDEGS.csv', index=True)
gene1b_data.to_csv('Genes_mutA_allDEGS.csv', index=True)
gene1c_data.to_csv('Genes_mutB_allDEGS.csv', index=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming selected_columns is one DataFrame and selected_columns_sigs is another

# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(res2['baseMean']), res2['log2FoldChange'], alpha=0.5, label='Total Genes', color='gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(Control_sigs['baseMean']), Control_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - Control')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - Control Total Genes & DEGs.jpg')

# Show the plot
plt.show()


# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(res3['baseMean']), res3['log2FoldChange'], alpha=0.5, label='Total Genes', color='gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(MutA_sigs['baseMean']), MutA_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - MutA')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - MutA Total Genes & DEGs.jpg')

# Show the plot
plt.show()


# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(res4['baseMean']), res4['log2FoldChange'], alpha=0.5, label='Total Genes', color='Gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(MutB_sigs['baseMean']), MutB_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - MutB')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - MutB Total Genes & DEGs.jpg')

# Show the plot
plt.show()


# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(res1['baseMean']), res1['log2FoldChange'], alpha=0.5, label='Total Genes', color='Gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(MutA_vs_MutB_vehicle_sigs['baseMean']), MutA_vs_MutB_vehicle_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - MutA vs MutB Vehicle')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - MutA vs MutB Vehicle Total Genes & DEGs.jpg')

# Show the plot
plt.show()

In [ ]:
# Assuming selected_columns_sigs is your DataFrame

# Create masks for upregulated and downregulated genes based on the magnitude of the 'stat' column
Control_upregulated_genes_mask = Control_sigs['stat'] > 0
Control_downregulated_genes_mask = Control_sigs['stat'] < 0

# Extract upregulated and downregulated genes
Control_upregulated_genes = Control_sigs.loc[Control_upregulated_genes_mask]
Control_downregulated_genes = Control_sigs.loc[Control_downregulated_genes_mask]

# Print the upregulated and downregulated genes
print("Number of Upregulated Genes - Control:", len(Control_upregulated_genes))
print("\nControl Upregulated Genes:")
print(Control_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

# Save upregulated genes to CSV
Control_upregulated_genes.to_csv('Control_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - Control:", len(Control_downregulated_genes))
print("\nControl Downregulated Genes:")
print(Control_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

# Save downregulated genes to CSV
Control_downregulated_genes.to_csv('Control_downregulated_genes.csv', index=True)

# Assuming selected_columns_sigs is your DataFrame

# MutA
MutA_upregulated_genes_mask = MutA_sigs['stat'] > 0
MutA_downregulated_genes_mask = MutA_sigs['stat'] < 0

MutA_upregulated_genes = MutA_sigs.loc[MutA_upregulated_genes_mask]
MutA_downregulated_genes = MutA_sigs.loc[MutA_downregulated_genes_mask]

print("Number of Upregulated Genes - MutA:", len(MutA_upregulated_genes))
print("\nMutA Upregulated Genes:")
print(MutA_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

MutA_upregulated_genes.to_csv('MutA_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - MutA:", len(MutA_downregulated_genes))
print("\nMutA Downregulated Genes:")
print(MutA_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

MutA_downregulated_genes.to_csv('MutA_downregulated_genes.csv', index=True)


# MutB
MutB_upregulated_genes_mask = MutB_sigs['stat'] > 0
MutB_downregulated_genes_mask = MutB_sigs['stat'] < 0

MutB_upregulated_genes = MutB_sigs.loc[MutB_upregulated_genes_mask]
MutB_downregulated_genes = MutB_sigs.loc[MutB_downregulated_genes_mask]

print("Number of Upregulated Genes - MutB:", len(MutB_upregulated_genes))
print("\nMutB Upregulated Genes:")
print(MutB_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

MutB_upregulated_genes.to_csv('MutB_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - MutB:", len(MutB_downregulated_genes))
print("\nMutB Downregulated Genes:")
print(MutB_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

MutB_downregulated_genes.to_csv('MutB_downregulated_genes.csv', index=True)


# MutA vs MutB Vehicle
MutA_vs_MutB_Vehicle_upregulated_genes_mask = MutA_vs_MutB_vehicle_sigs['stat'] > 0
MutA_vs_MutB_Vehicle_downregulated_genes_mask = MutA_vs_MutB_vehicle_sigs['stat'] < 0

MutA_vs_MutB_Vehicle_upregulated_genes = MutA_vs_MutB_vehicle_sigs.loc[MutA_vs_MutB_Vehicle_upregulated_genes_mask]
MutA_vs_MutB_Vehicle_downregulated_genes = MutA_vs_MutB_vehicle_sigs.loc[MutA_vs_MutB_Vehicle_downregulated_genes_mask]

print("Number of Upregulated Genes - MutA v/s MutB Vehicle:", len(MutA_vs_MutB_Vehicle_upregulated_genes))
print("\nMutA v/s MutB Vehicle Upregulated Genes:")
print(MutA_vs_MutB_Vehicle_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

MutA_vs_MutB_Vehicle_upregulated_genes.to_csv('MutA_vs_MutB_Vehicle_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - MutA v/s MutB Vehicle :", len(MutA_vs_MutB_Vehicle_downregulated_genes))
print("\nMutA v/s MutB Vehicle  Downregulated Genes:")
print(MutA_vs_MutB_Vehicle_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

MutA_vs_MutB_Vehicle_downregulated_genes.to_csv('MutA_vs_MutB_Vehicle_downregulated_genes.csv', index=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

# Assuming 'Symbol' column contains gene symbols in your DataFrames

# Extracting all columns from each set of significant genes
geneup1_data = Control_upregulated_genes.copy()
geneup2_data = MutA_upregulated_genes.copy()
geneup3_data = MutB_upregulated_genes.copy()

# Calculate the total number of unique genes
total_genes_sigs = len(set.union(set(geneup1_data.index), set(geneup2_data.index), set(geneup3_data.index)))

# Create a Venn diagram with equal-sized circles
venn_labels_sigs = {'100': len(set(geneup1_data.index) - set(geneup2_data.index) - set(geneup3_data.index)),
                    '010': len(set(geneup2_data.index) - set(geneup1_data.index) - set(geneup3_data.index)),
                    '001': len(set(geneup3_data.index) - set(geneup1_data.index) - set(geneup2_data.index)),
                    '110': len(set(geneup1_data.index) & set(geneup2_data.index) - set(geneup3_data.index)),
                    '101': len(set(geneup1_data.index) & set(geneup3_data.index) - set(geneup2_data.index)),
                    '011': len(set(geneup2_data.index) & set(geneup3_data.index) - set(geneup1_data.index)),
                    '111': len(set(geneup1_data.index) & set(geneup2_data.index) & set(geneup3_data.index))
                   }

venn_labels_sigs = {key: f"{value}" for key, value in venn_labels_sigs.items()}

# Define custom colors for each set
custom_colors = {
    '100': 'green',   # Control
    '010': 'red',  # MutA
    '001': 'blue',    # MutB
}


# Create a DataFrame for each set of genes
Genes_control_upDEGs = geneup1_data.loc[~geneup1_data.index.isin(set(geneup2_data.index) | set(geneup3_data.index))]
Genes_mutA_upDEGs = geneup2_data.loc[~geneup2_data.index.isin(set(geneup1_data.index) | set(geneup3_data.index))]
Genes_mutB_upDEGs = geneup3_data.loc[~geneup3_data.index.isin(set(geneup1_data.index) | set(geneup2_data.index))]
Genes_control_mutA_upDEGs = geneup1_data.loc[geneup1_data.index.isin(set(geneup2_data.index).difference(set(geneup3_data.index)))]
Genes_mutB_mutA_upDEGs = geneup2_data.loc[geneup2_data.index.isin(set(geneup3_data.index).difference(set(geneup1_data.index)))]
Genes_control_mutB_upDEGs = geneup1_data.loc[geneup1_data.index.isin(set(geneup3_data.index).difference(set(geneup2_data.index)))]
Common_Genes_upDEGs = pd.DataFrame({"Genes": list(set(geneup1_data.index) & set(geneup2_data.index) & set(geneup3_data.index))})

# Display or use the extracted genes as needed
print("Genes only expressed in Control:")
print(Genes_control_upDEGs)

print("\nGenes only expressed in MutA:")
print(Genes_mutA_upDEGs)

print("\nGenes only expressed in MutB:")
print(Genes_mutB_upDEGs)

print("\nGenes expressed between Control and MutA:")
print(Genes_control_mutA_upDEGs)

print("\nGenes expressed between MutB and MutA:")
print(Genes_mutB_mutA_upDEGs)

print("\nGenes expressed between Control and MutB:")
print(Genes_control_mutB_upDEGs)

print("\nGenes expressed between Control, MutB, and MutA:")
print(Common_Genes_upDEGs)

# Save genes to CSV
Genes_control_upDEGs.to_csv('Genes_control_upDEGs.csv', index=True)
Genes_mutA_upDEGs.to_csv('Genes_mutA_upDEGs.csv', index=True)
Genes_mutB_upDEGs.to_csv('Genes_mutB_upDEGs.csv', index=True)
Genes_control_mutA_upDEGs.to_csv('Genes_control_mutA_upDEGs.csv', index=True)
Genes_mutB_mutA_upDEGs.to_csv('Genes_mutB_mutA_upDEGs.csv', index=True)
Genes_control_mutB_upDEGs.to_csv('Genes_control_mutB_upDEGs.csv', index=True)
Common_Genes_upDEGs.to_csv('Common_Genes_upDEGs.csv', index=True)

# Create a Venn diagram
venn_DEGs = venn3([set(geneup1_data.index), set(geneup2_data.index), set(geneup3_data.index)],
                  set_labels=('Control', 'MutA', 'MutB'),
                  normalize_to=total_genes_sigs)

# Set custom colors for each set
for key, value in custom_colors.items():
    venn_DEGs.get_patch_by_id(key).set_color(value)

# Set labels for each region in the Venn diagram
for key, value in venn_labels_sigs.items():
    text = venn_DEGs.get_label_by_id(key)
    text.set_text(value)
    text.set_fontweight('bold')  # set text to bold

# Set all text in the plot to bold
for text in venn_DEGs.set_labels + venn_DEGs.subset_labels:
    text.set_fontweight('bold')

# Display the plot
plt.title("Venn Diagram - Upregulated DEGs", y=1.08, fontweight='bold')

plt.savefig('Venn Diagram - Upregulated DEGs.jpg')

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

# Assuming 'Symbol' column contains gene symbols in your DataFrames

# Extracting all columns from each set of significant genes
genedown1_data = Control_downregulated_genes.copy()
genedown2_data = MutA_downregulated_genes.copy()
genedown3_data = MutB_downregulated_genes.copy()

# Calculate the total number of unique genes
total_genes_sigs = len(set.union(set(genedown1_data.index), set(genedown2_data.index), set(genedown3_data.index)))

# Create a Venn diagram with equal-sized circles
venn_labels_sigs = {'100': len(set(genedown1_data.index) - set(genedown2_data.index) - set(genedown3_data.index)),
                    '010': len(set(genedown2_data.index) - set(genedown1_data.index) - set(genedown3_data.index)),
                    '001': len(set(genedown3_data.index) - set(genedown1_data.index) - set(genedown2_data.index)),
                    '110': len(set(genedown1_data.index) & set(genedown2_data.index) - set(genedown3_data.index)),
                    '101': len(set(genedown1_data.index) & set(genedown3_data.index) - set(genedown2_data.index)),
                    '011': len(set(genedown2_data.index) & set(genedown3_data.index) - set(genedown1_data.index)),
                    '111': len(set(genedown1_data.index) & set(genedown2_data.index) & set(genedown3_data.index))
                   }

venn_labels_sigs = {key: f"{value}" for key, value in venn_labels_sigs.items()}

# Define custom colors for each set
custom_colors = {
    '100': 'green',   # Control
    '010': 'red',  # MutA
    '001': 'blue',    # MutB
}


# Create a DataFrame for each set of genes
Genes_control_downDEGs = genedown1_data.loc[~genedown1_data.index.isin(set(genedown2_data.index) | set(genedown3_data.index))]
Genes_mutA_downDEGs = genedown2_data.loc[~genedown2_data.index.isin(set(genedown1_data.index) | set(genedown3_data.index))]
Genes_mutB_downDEGs = genedown3_data.loc[~genedown3_data.index.isin(set(genedown1_data.index) | set(genedown2_data.index))]
Genes_control_mutA_downDEGs = genedown1_data.loc[genedown1_data.index.isin(set(genedown2_data.index).difference(set(genedown3_data.index)))]
Genes_mutB_mutA_downDEGs = genedown2_data.loc[genedown2_data.index.isin(set(genedown3_data.index).difference(set(genedown1_data.index)))]
Genes_control_mutB_downDEGs = genedown1_data.loc[genedown1_data.index.isin(set(genedown3_data.index).difference(set(genedown2_data.index)))]
Common_Genes_downDEGs = pd.DataFrame({"Genes": list(set(genedown1_data.index) & set(genedown2_data.index) & set(genedown3_data.index))})

# Display or use the extracted genes as needed
print("Genes only expressed in Control:")
print(Genes_control_downDEGs)

print("\nGenes only expressed in MutA:")
print(Genes_mutA_downDEGs)

print("\nGenes only expressed in MutB:")
print(Genes_mutB_downDEGs)

print("\nGenes expressed between Control and MutA:")
print(Genes_control_mutA_downDEGs)

print("\nGenes expressed between MutB and MutA:")
print(Genes_mutB_mutA_downDEGs)

print("\nGenes expressed between Control and MutB:")
print(Genes_control_mutB_downDEGs)

print("\nGenes expressed between Control, MutB, and MutA:")
print(Common_Genes_downDEGs)

# Save genes to CSV
Genes_control_downDEGs.to_csv('Genes_control_downDEGs.csv', index=True)
Genes_mutA_downDEGs.to_csv('Genes_mutA_downDEGs.csv', index=True)
Genes_mutB_downDEGs.to_csv('Genes_mutB_downDEGs.csv', index=True)
Genes_control_mutA_downDEGs.to_csv('Genes_control_mutA_downDEGs.csv', index=True)
Genes_mutB_mutA_downDEGs.to_csv('Genes_mutB_mutA_downDEGs.csv', index=True)
Genes_control_mutB_downDEGs.to_csv('Genes_control_mutB_downDEGs.csv', index=True)
Common_Genes_downDEGs.to_csv('Common_Genes_downDEGs.csv', index=True)

# Create a Venn diagram
venn_DEGs = venn3([set(genedown1_data.index), set(genedown2_data.index), set(genedown3_data.index)],
                  set_labels=('Control', 'MutA', 'MutB'),
                  normalize_to=total_genes_sigs)

# Set custom colors for each set
for key, value in custom_colors.items():
    venn_DEGs.get_patch_by_id(key).set_color(value)

# Set labels for each region in the Venn diagram
for key, value in venn_labels_sigs.items():
    text = venn_DEGs.get_label_by_id(key)
    text.set_text(value)
    text.set_fontweight('bold')  # set text to bold

# Set all text in the plot to bold
for text in venn_DEGs.set_labels + venn_DEGs.subset_labels:
    text.set_fontweight('bold')

# Display the plot
plt.title("Venn Diagram - Downregulated DEGs", y=1.08, fontweight='bold')

plt.savefig('Venn Diagram - Downregulated DEGs.jpg')

plt.show()

In [ ]:
Genes_control_downDEGs

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
Control_only_enr_GOBP_up1 = gp.enrichr(
    gene_list=Genes_control_upDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='Control_only_enr_GOBP_up',
    cutoff=0.5
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
Control_only_enr_GOBP_down1 = gp.enrichr(
    gene_list=Genes_control_downDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='Control_only_enr_GOBP_down',
    cutoff=0.5
)



# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(Control_only_enr_GOBP_up1.results, title="Control(only): Enriched GOBP (Upregulated Genes)", cutoff=0.5, color='red')
plt.savefig('Control (only) : Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(Control_only_enr_GOBP_down1.results, title="Control(only): Enriched GOBP (Downregulated Genes)", cutoff=0.5, color='blue')
plt.savefig('Control (only): Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results for upregulated genes
Control_only_enrichment_results_up = Control_only_enr_GOBP_up1.results

# Convert the results to a DataFrame
Control_only_df_upregulated = pd.DataFrame(Control_only_enrichment_results_up)

# Save the DataFrame to a CSV file with "Control" in the filename
Control_only_df_upregulated.to_csv('Control_only_enr_GOBP_up1.csv', index=False)

# Extracting relevant information from the results for downregulated genes
Control_only_enrichment_results_down = Control_only_enr_GOBP_down1.results

# Convert the results to a DataFrame
Control_only_df_downregulated = pd.DataFrame(Control_only_enrichment_results_down)

# Save the DataFrame to a CSV file with "Control" in the filename
Control_only_df_downregulated.to_csv('Control_only_enr_GOBP_down1.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
Control_enr_GOBP_up1 = gp.enrichr(
    gene_list=Control_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='Control_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
Control_enr_GOBP_down1 = gp.enrichr(
    gene_list=Control_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='Control_enr_GOBP_down',
    cutoff=0.05
)



# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(Control_enr_GOBP_up1.results, title="Control: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('Control: Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(Control_enr_GOBP_down1.results, title="Control: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('Control: Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
MutA_only_enr_GOBP_up1 = gp.enrichr(
    gene_list=Genes_mutA_upDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutA_only_enr_GOBP_up',
    cutoff=0.25
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
MutA_only_enr_GOBP_down1 = gp.enrichr(
    gene_list=Genes_mutA_downDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutA_only_enr_GOBP_down',
    cutoff=0.25
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(MutA_only_enr_GOBP_up1.results, title="MutA(only): Enriched GOBP (Upregulated Genes)", cutoff=0.25, color='red')
plt.savefig('MutA (only) : Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(MutA_only_enr_GOBP_down1.results, title="MutA(only): Enriched GOBP (Downregulated Genes)", cutoff=0.25, color='blue')
plt.savefig('MutA (only): Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results
MutA_only_enrichment_results_up = MutA_only_enr_GOBP_up1.results

# Convert the results to a DataFrame
MutA_only_df_upregulated = pd.DataFrame(MutA_only_enrichment_results_up)

# Save the DataFrame to a CSV file
MutA_only_df_upregulated.to_csv('MutA_only_enr_GOBP_up1.csv', index=False)

import pandas as pd

# Extracting relevant information from the results
MutA_only_enrichment_results_down = MutA_only_enr_GOBP_down1.results

# Convert the results to a DataFrame
MutA_only_df_downregulated = pd.DataFrame(MutA_only_enrichment_results_down)

# Save the DataFrame to a CSV file
MutA_only_df_downregulated.to_csv('MutA_only_enr_GOBP_down1.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
MutA_enr_GOBP_up1 = gp.enrichr(
    gene_list=MutA_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutA_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
MutA_enr_GOBP_down1 = gp.enrichr(
    gene_list=MutA_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutA_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(MutA_enr_GOBP_up1.results, title="MutA: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('MutA_Enriched_GOBP_Upregulated_Genes.jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(MutA_enr_GOBP_down1.results, title="MutA: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('MutA_Enriched_GOBP_Downregulated_Genes.jpg', bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
MutB_only_enr_GOBP_up1 = gp.enrichr(
    gene_list=Genes_mutB_upDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutB_only_enr_GOBP_up',
    cutoff=0.25
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
MutB_only_enr_GOBP_down1 = gp.enrichr(
    gene_list=Genes_mutB_downDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutB_only_enr_GOBP_down',
    cutoff=0.25
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(MutB_only_enr_GOBP_up1.results, title="MutB (only): Enriched GOBP (Upregulated Genes)", cutoff=0.25, color='red')
plt.savefig('MutB (only) : Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(MutB_only_enr_GOBP_down1.results, title="MutB (only): Enriched GOBP (Downregulated Genes)", cutoff=0.25, color='blue')
plt.savefig('MutB (only): Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results for upregulated genes
MutB_only_enrichment_results_up = MutB_only_enr_GOBP_up1.results

# Convert the results to a DataFrame
MutB_only_df_upregulated = pd.DataFrame(MutB_only_enrichment_results_up)

# Save the DataFrame to a CSV file with "MutB" in the filename
MutB_only_df_upregulated.to_csv('MutB_only_enr_GOBP_up1.csv', index=False)

# Extracting relevant information from the results for downregulated genes
MutB_only_enrichment_results_down = MutB_only_enr_GOBP_down1.results

# Convert the results to a DataFrame
MutB_only_df_downregulated = pd.DataFrame(MutB_only_enrichment_results_down)

# Save the DataFrame to a CSV file with "MutB" in the filename
MutB_only_df_downregulated.to_csv('MutB_only_enr_GOBP_down1.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
MutB_enr_GOBP_up1 = gp.enrichr(
    gene_list=MutB_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutB_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
MutB_enr_GOBP_down1 = gp.enrichr(
    gene_list=MutB_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='MutB_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(MutB_enr_GOBP_up1.results, title="MutB: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('MutB_Enriched_GOBP_Upregulated_Genes.jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(MutB_enr_GOBP_down1.results, title="MutB: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('MutB_Enriched_GOBP_Downregulated_Genes.jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results for upregulated genes
MutB_enrichment_results_up = MutB_enr_GOBP_up1.results

# Convert the results to a DataFrame
MutB_df_upregulated = pd.DataFrame(MutB_enrichment_results_up)

# Save the DataFrame to a CSV file with "MutB" in the filename
MutB_df_upregulated.to_csv('MutB_enr_GOBP_up1.csv', index=False)

# Extracting relevant information from the results for downregulated genes
MutB_o_enrichment_results_down = MutB_only_enr_GOBP_down1.results

# Convert the results to a DataFrame
MutB_only_df_downregulated = pd.DataFrame(MutB_only_enrichment_results_down)

# Save the DataFrame to a CSV file with "MutB" in the filename
MutB_only_df_downregulated.to_csv('MutB_only_enr_GOBP_down1.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
CarvsMutB_enr_GOBP_up1 = gp.enrichr(
    gene_list=MutA_vs_MutB_Vehicle_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='CarvsMutB_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
CarvsMutB_enr_GOBP_down1 = gp.enrichr(
    gene_list=MutA_vs_MutB_Vehicle_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='CarvsMutB_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(CarvsMutB_enr_GOBP_up1.results, title="CarvsMutB: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('CarvsMutB_Enriched_GOBP_Upregulated_Genes.jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(CarvsMutB_enr_GOBP_down1.results, title="CarvsMutB: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('CarvsMutB_Enriched_GOBP_Downregulated_Genes.jpg', bbox_inches="tight")

plt.show()


In [ ]:
stat_res2_rnk

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gseapy as gp

# Assuming Control_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
Control_rnk = stat_res2_rnk.copy()

# Sort the genes by rank in descending order
Control_rnk = Control_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
Control_ranking = Control_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
Control_ranking.to_csv('Control_ranking.csv', index=False)

# Perform gene set enrichment analysis
Control_pre_res = gp.prerank(rnk=Control_ranking, gene_sets='GO_Biological_Process_2023',
                                     seed=6, permutation_num=100, min_size=20, max_size=1000)

# Filter out terms with FDR q-value equal to zero
Control_pre_res.res2d = Control_pre_res.res2d[Control_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
Control_pre_res.res2d = Control_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
Control_terms = Control_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
Control_axs = Control_pre_res.plot(terms=Control_terms,
                                              show_ranking=True,
                                              figsize=(3, 4))

# Save the plot
plt.savefig('Control_rank.jpg', bbox_inches="tight")

Control_pre_res.res2d.to_csv('Control_pre_res.csv')
Control_pre_res.res2d

In [ ]:
# Assuming MutA_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
MutA_rnk = stat_res3_rnk.copy()

# Sort the genes by rank in descending order
MutA_rnk = MutA_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
MutA_ranking = MutA_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
MutA_ranking.to_csv('MutA_ranking.csv', index=False)

# Perform gene set enrichment analysis
MutA_pre_res = gp.prerank(rnk=MutA_ranking, gene_sets='GO_Biological_Process_2023',
                             seed=6, permutation_num=100, min_size=20, max_size=1000)

# Filter out terms with FDR q-value equal to zero
MutA_pre_res.res2d = MutA_pre_res.res2d[MutA_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
MutA_pre_res.res2d = MutA_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
MutA_terms = MutA_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
MutA_axs = MutA_pre_res.plot(terms=MutA_terms,
                                   show_ranking=True,
                                   figsize=(3, 4))

# Save the plot
plt.savefig('MutA_rank.jpg', bbox_inches="tight")

MutA_pre_res.res2d.to_csv('MutA_pre_res.csv')
MutA_pre_res.res2d


In [ ]:
# Assuming MutB_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
MutB_rnk = stat_res2_rnk.copy()

# Sort the genes by rank in descending order
MutB_rnk = MutB_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
MutB_ranking = MutB_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
MutB_ranking.to_csv('MutB_ranking.csv', index=False)

# Perform gene set enrichment analysis
MutB_pre_res = gp.prerank(rnk=MutB_ranking, gene_sets='GO_Biological_Process_2023',
                             seed=6, permutation_num=100, min_size=20, max_size=1000)

# Filter out terms with FDR q-value equal to zero
MutB_pre_res.res2d = MutB_pre_res.res2d[MutB_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
MutB_pre_res.res2d = MutB_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
MutB_terms = MutB_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
MutB_axs = MutB_pre_res.plot(terms=MutB_terms,
                                   show_ranking=True,
                                   figsize=(3, 4))

# Save the plot
plt.savefig('MutB_rank.jpg', bbox_inches="tight")

MutB_pre_res.res2d.to_csv('MutB_pre_res.csv')
MutB_pre_res.res2d


In [ ]:
Normalised_dataset.head(2000000)

In [ ]:
pc3_sorted_df.head(10000000)

In [ ]:
Treated_PC3=pc3_sorted_df.abs()
Treated_PC3.head(100000)

In [ ]:
Norm_data_rearr = Normalised_dataset.reindex(Treated_PC3.index)
Norm_data_rearr

In [ ]:
PC3_weighted_Norm_dataset = Norm_data_rearr.mul(Treated_PC3.iloc[:, 0], axis=0)
PC3_weighted_Norm_dataset.to_csv('PC3_weighted_Norm_dataset.csv')
PC3_weighted_Norm_dataset.head(1000000)

In [ ]:
PC3_weighted_Norm_dataset1 = PC3_weighted_Norm_dataset.astype(int)
PC3_weighted_Norm_dataset1=PC3_weighted_Norm_dataset1.T
PC3_weighted_Norm_dataset1.head(1000000)

In [ ]:
metadata1 = pd.DataFrame(zip(PC3_weighted_Norm_dataset1.index, ['Control-Vehicle','Control-Vehicle','Control-Vehicle','Control-Treated','Control-Treated','Control-Treated','MutA-Vehicle', 'MutA-Vehicle', 'MutA-Vehicle','MutA-Treated', 'MutA-Treated', 'MutA-Treated', 'MutB-Vehicle', 'MutB-Vehicle', 'MutB-Vehicle','MutB-Treated', 'MutB-Treated', 'MutB-Treated']),
                        columns = ['Sample', 'Condition'])
metadata1 = metadata1.set_index('Sample')
metadata1

In [ ]:
dds1 = DeseqDataSet(counts=PC3_weighted_Norm_dataset1,
            metadata=metadata1,
            design_factors=["Condition"])

In [ ]:
dds1.deseq2()

In [ ]:
dds1

In [ ]:
PC3_stat_res1 = DeseqStats(dds1, contrast = ('Condition','Control-Treated','Control-Vehicle'))
PC3_stat_res1.summary()
PC3_stat_res1=PC3_stat_res1.results_df
PC3_stat_res1=pd.merge(PC3_stat_res1, data2[['gene_name']], left_index=True, right_index=True)
PC3_stat_res1

In [ ]:
# Copy the downregulated genes data
PC3_stat_res1_rnk = PC3_stat_res1.copy()

# Calculate the rank based on adjusted p-value
PC3_stat_res1_rnk['Rank'] = PC3_stat_res1.stat

# Sort the genes by rank in descending order
PC3_stat_res1_rnk = PC3_stat_res1_rnk.sort_values('Rank', ascending=False)
PC3_stat_res1_rnk


In [ ]:
PC3_res1 = PC3_stat_res1_rnk
PC3_res1


In [ ]:
PC3_Control_sigs = PC3_res1[(PC3_res1.padj < 0.05)]
PC3_Control_sigs.to_csv('PC3_Control_sigs.csv', index=True)
PC3_Control_sigs

In [ ]:
PC3_stat_res2 = DeseqStats(dds1, contrast = ('Condition','MutA-Treated','MutA-Vehicle'))
PC3_stat_res2.summary()
PC3_stat_res2=PC3_stat_res2.results_df
PC3_stat_res2=pd.merge(PC3_stat_res2, data2[['gene_name']], left_index=True, right_index=True)
PC3_stat_res2

In [ ]:
# Copy the downregulated genes data
PC3_stat_res2_rnk = PC3_stat_res2.copy()

# Calculate the rank based on adjusted p-value
PC3_stat_res2_rnk['Rank'] = PC3_stat_res2.stat

# Sort the genes by rank in descending order
PC3_stat_res2_rnk = PC3_stat_res2_rnk.sort_values('Rank', ascending=False)
PC3_stat_res2_rnk


In [ ]:
PC3_res2 = PC3_stat_res2_rnk
PC3_res2 = PC3_res2[PC3_res2['padj'] != 0]
PC3_res2

In [ ]:
PC3_MutA_sigs = PC3_res2[(PC3_res2.padj < 0.05)]
PC3_MutA_sigs.to_csv('PC3_MutA_sigs.csv', index=True)
PC3_MutA_sigs

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Assuming PC3_stat_res2_rnk is your DataFrame with columns: 'gene_name', 'log2FoldChange', 'padj'
# Replace this with the actual loading code if it's from a file or another source
# For example, if it's from a CSV:
# PC3_stat_res2_rnk = pd.read_csv('path_to_your_file.csv')

# Calculate -log10(padj)
PC3_res2['-log10(padj)'] = -np.log10(PC3_res2['padj'])

# Determine the thresholds for significance
log2fc_threshold = 0.5
padj_threshold = 0.05

# Define significant genes and their direction
PC3_res2['up_regulated'] = (PC3_res2['log2FoldChange'] > log2fc_threshold) & (PC3_res2['padj'] < padj_threshold)
PC3_res2['down_regulated'] = (PC3_res2['log2FoldChange'] < -log2fc_threshold) & (PC3_res2['padj'] < padj_threshold)
PC3_res2['significant'] = PC3_res2['up_regulated'] | PC3_res2['down_regulated']

# Create the volcano plot
plt.figure(figsize=(10, 8))
sns.scatterplot(data=PC3_res2, x='log2FoldChange', y='-log10(padj)', 
                hue='significant', palette={True: 'grey', False: 'grey'}, legend=None)
sns.scatterplot(data=PC3_res2[PC3_res2['up_regulated']], x='log2FoldChange', y='-log10(padj)', color='red', legend=None)
sns.scatterplot(data=PC3_res2[PC3_res2['down_regulated']], x='log2FoldChange', y='-log10(padj)', color='blue', legend=None)

# Add horizontal and vertical lines for thresholds
plt.axhline(y=-np.log10(padj_threshold), color='gray', linestyle='--')
plt.axvline(x=log2fc_threshold, color='gray', linestyle='--')
plt.axvline(x=-log2fc_threshold, color='gray', linestyle='--')

# Highlight top-ranked genes
num_genes_to_label = 6  # Define number of genes to label in each category

# Get top up-regulated and down-regulated genes
top_up_genes = PC3_res2[PC3_res2['up_regulated']].sort_values(by='-log10(padj)', ascending=False).head(num_genes_to_label)
top_down_genes = PC3_res2[PC3_res2['down_regulated']].sort_values(by='-log10(padj)', ascending=False).head(num_genes_to_label)

# Concatenate top genes for labeling
top_genes = pd.concat([top_up_genes, top_down_genes])

for i in range(top_genes.shape[0]):
    plt.text(top_genes['log2FoldChange'].iloc[i], top_genes['-log10(padj)'].iloc[i],
             top_genes['gene_name'].iloc[i], horizontalalignment='left', size='medium', color='black', weight='semibold')

plt.xlabel('log2(Fold Change)')
plt.ylabel('-log10(p-adjusted)')
plt.title('Volcano Plot')
plt.show()


In [ ]:
data.head()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage

# List of specific genes of interest
genes_of_interest = ['ENSG00000277586', 'ENSG00000104722', 'ENSG00000111640', 'ENSG00000132718', 'ENSG00000197747', 'ENSG00000067057', 'ENSG00000177606', 'ENSG00000100906', 'ENSG00000163873', 'ENSG00000104435', 'ENSG00000155980', 'ENSG00000132639', 'ENSG00000171885', 'ENSG00000131095', 'ENSG00000112096', 'ENSG00000164326', 'ENSG00000093010', 'ENSG00000249992', 'ENSG00000130203', 'ENSG00000142192', 'ENSG00000106236',  'ENSG00000221890','ENSG00000171246', 'ENSG00000102003', 'ENSG00000011347', 'ENSG00000206503', 'ENSG00000147894', 'ENSG00000019505', 'ENSG00000057663', 'ENSG00000198793', 'ENSG00000089280', 'ENSG00000137601', 'ENSG00000277586']

# Subset the data frame
df_subset = PC3_weighted_Norm_dataset.loc[genes_of_interest]
df_subset=pd.merge(df_subset, data2[['gene_name']], left_index=True, right_index=True)
df_subset.set_index('gene_name', inplace=True)
df_subset

In [ ]:
# Define the colors for the first set of samples
samples = {
    "OrgB_Control_Vehicle_n1": "lightgreen", "OrgB_Control_Vehicle_n2": "lightgreen", "OrgB_Control_Vehicle_n3": "lightgreen",
    "OrgB_Control_Treated_n1": "green", "OrgB_Control_Treated_n2": "green", "OrgB_Control_Treated_n3": "green",
    "OrgB_MutA_Vehicle_n1": "lightcoral", "OrgB_MutA_Vehicle_n2": "lightcoral", "OrgB_MutA_Vehicle_n3": "lightcoral",
    "OrgB_MutA_Treated_n1": "red", "OrgB_MutA_Treated_n2": "red", "OrgB_MutA_Treated_n3": "red",
    "OrgB_MutB_Vehicle_n1": "lightblue", "OrgB_MutB_Vehicle_n2": "lightblue", "OrgB_MutB_Vehicle_n3": "lightblue",
    "OrgB_MutB_Treated_n1": "blue", "OrgB_MutB_Treated_n2": "blue", "OrgB_MutB_Treated_n3": "blue",
}

# Define the colors for the second set of samples
#samples1 = { 
    #"OrgB_Control_Vehicle_n1": "gray", "OrgB_Control_Vehicle_n2": "gray", "OrgB_Control_Vehicle_n3": "gray",
  #  "OrgB_Control_Treated_n1": "MediumVioletRed", "OrgB_Control_Treated_n2": "MediumVioletRed", "OrgB_Control_Treated_n3": "MediumVioletRed",
   # "OrgB_MutA_Vehicle_n1": "gray", "OrgB_MutA_Vehicle_n2": "gray", "OrgB_MutA_Vehicle_n3": "gray",
    #"OrgB_MutA_Treated_n1": "MediumVioletRed", "OrgB_MutA_Treated_n2": "MediumVioletRed", "OrgB_MutA_Treated_n3": "MediumVioletRed",
    #"OrgB_MutB_Vehicle_n1": "gray", "OrgB_MutB_Vehicle_n2": "gray", "OrgB_MutB_Vehicle_n3": "gray",
    #"OrgB_MutB_Treated_n1": "MediumVioletRed", "OrgB_MutB_Treated_n2": "MediumVioletRed", "OrgB_MutB_Treated_n3": "MediumVioletRed",
#}

# Generate the column colors for both sets
column_colors = [samples[col] for col in df_subset.columns]
#column_colors1 = [samples1[col] for col in df_subset.columns]


# Create the clustermap
plot4 = sns.clustermap(df_subset, 
                       z_score=0, 
                       cmap="vlag", 
                       figsize=(10, 20),
                       vmin=-1, 
                       vmax=1, 
                       cbar_kws={"label": "z score"},
                       col_colors=[column_colors],
                       cbar_pos=(1, 0.75, 0.05, 0.15), 
                       xticklabels=False)

# Access the dendrogram LineCollection objects and set their properties
for dendrogram in [plot4.ax_row_dendrogram, plot4.ax_col_dendrogram]:
    for line in dendrogram.collections:
        line.set_linewidths(3)  # Set the desired line width


# Set x tick labels color to white
plot4.ax_heatmap.tick_params(axis='x', colors='white')

# Increase the font size of color bar tick labels
plot4.cax.set_yticklabels(plot4.cax.get_yticklabels(), fontsize=15) 

# Increase the font size of color bar labels
plot4.cax.set_ylabel('z score', fontsize=15)  # Adjust 12 to your desired font size

# Define legend labels for specific columns of samples
legend_labels = {
    "OrgB_Control_Vehicle_n1": "Control Vehicle", "OrgB_Control_Treated_n1": "Control Treated", "OrgB_MutA_Vehicle_n1": "MutA Vehicle",  "OrgB_MutA_Treated_n1": "MutA Treated", "OrgB_MutB_Vehicle_n1": "MutB Vehicle",  "OrgB_MutB_Treated_n1": "MutB Treated"
}

# Define legend labels for specific columns of samples1
#legend_labels1 = {
    #"OrgB_Control_Vehicle_n1": "Vehicle", "OrgB_Control_Treated_n1": "Treated"}

# Create the first legend (for samples)
handles1 = [plt.Rectangle((0,0),1,1, color=samples[col]) for col in legend_labels.keys()]
legend1 = plot4.ax_col_dendrogram.legend(handles=handles1, labels=legend_labels.values(),
                                         loc='upper left', bbox_to_anchor=(1.1, -0.75),fontsize='xx-large',  frameon=False)

# Create the second legend (for samples1)
#handles2 = [plt.Rectangle((0,0),1,1, color=samples1[col]) for col in legend_labels1.keys()]
#legend2 = plot4.ax_col_dendrogram.legend(handles=handles2, labels=legend_labels1.values(),
                                        # loc='upper left', bbox_to_anchor=(1.1, -0.45), fontsize='xx-large',  frameon=False)

# Add the legends to the plot
plot4.ax_col_dendrogram.add_artist(legend1)
#plot4.ax_col_dendrogram.add_artist(legend2)


plt.savefig('Clustermap Selected genes.png', bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage

# List of specific genes of interest
genes_of_interest = ['ENSG00000277586', 'ENSG00000104722', 'ENSG00000132718', 'ENSG00000197747', 'ENSG00000067057', 'ENSG00000177606', 'ENSG00000100906', 'ENSG00000163873', 'ENSG00000104435', 'ENSG00000155980', 'ENSG00000132639', 'ENSG00000171885', 'ENSG00000131095', 'ENSG00000112096', 'ENSG00000164326', 'ENSG00000093010', 'ENSG00000249992', 'ENSG00000206503']

# Subset the data frame
df_subset = data.loc[genes_of_interest]
df_subset=pd.merge(df_subset, data2[['gene_name']], left_index=True, right_index=True)
df_subset.set_index('gene_name', inplace=True)
df_subset

In [ ]:


# Define the colors for the first set of samples
samples = {
    "OrgB_Control_Vehicle_n1": "lightgreen", "OrgB_Control_Vehicle_n2": "lightgreen", "OrgB_Control_Vehicle_n3": "lightgreen",
    "OrgB_Control_Treated_n1": "green", "OrgB_Control_Treated_n2": "green", "OrgB_Control_Treated_n3": "green",
    "OrgB_MutA_Vehicle_n1": "lightcoral", "OrgB_MutA_Vehicle_n2": "lightcoral", "OrgB_MutA_Vehicle_n3": "lightcoral",
    "OrgB_MutA_Treated_n1": "red", "OrgB_MutA_Treated_n2": "red", "OrgB_MutA_Treated_n3": "red",
    "OrgB_MutB_Vehicle_n1": "lightblue", "OrgB_MutB_Vehicle_n2": "lightblue", "OrgB_MutB_Vehicle_n3": "lightblue",
    "OrgB_MutB_Treated_n1": "blue", "OrgB_MutB_Treated_n2": "blue", "OrgB_MutB_Treated_n3": "blue",
}

# Define the colors for the second set of samples
#samples1 = { 
    #"OrgB_Control_Vehicle_n1": "gray", "OrgB_Control_Vehicle_n2": "gray", "OrgB_Control_Vehicle_n3": "gray",
  #  "OrgB_Control_Treated_n1": "MediumVioletRed", "OrgB_Control_Treated_n2": "MediumVioletRed", "OrgB_Control_Treated_n3": "MediumVioletRed",
   # "OrgB_MutA_Vehicle_n1": "gray", "OrgB_MutA_Vehicle_n2": "gray", "OrgB_MutA_Vehicle_n3": "gray",
    #"OrgB_MutA_Treated_n1": "MediumVioletRed", "OrgB_MutA_Treated_n2": "MediumVioletRed", "OrgB_MutA_Treated_n3": "MediumVioletRed",
    #"OrgB_MutB_Vehicle_n1": "gray", "OrgB_MutB_Vehicle_n2": "gray", "OrgB_MutB_Vehicle_n3": "gray",
    #"OrgB_MutB_Treated_n1": "MediumVioletRed", "OrgB_MutB_Treated_n2": "MediumVioletRed", "OrgB_MutB_Treated_n3": "MediumVioletRed",
#}

# Generate the column colors for both sets
column_colors = [samples[col] for col in df_subset.columns]
#column_colors1 = [samples1[col] for col in df_subset.columns]


# Create the clustermap
plot4 = sns.clustermap(df_subset, 
                       z_score=0, 
                       cmap="vlag", 
                       figsize=(10, 20),
                       vmin=-1, 
                       vmax=1, 
                       cbar_kws={"label": "z score"},
                       col_colors=[column_colors],
                       cbar_pos=(1, 0.75, 0.05, 0.15), 
                       xticklabels=False)

# Access the dendrogram LineCollection objects and set their properties
for dendrogram in [plot4.ax_row_dendrogram, plot4.ax_col_dendrogram]:
    for line in dendrogram.collections:
        line.set_linewidths(3)  # Set the desired line width


# Set x tick labels color to white
plot4.ax_heatmap.tick_params(axis='x', colors='white')

# Increase the font size of color bar tick labels
plot4.cax.set_yticklabels(plot4.cax.get_yticklabels(), fontsize=15) 

# Increase the font size of color bar labels
plot4.cax.set_ylabel('z score', fontsize=15)  # Adjust 12 to your desired font size

# Define legend labels for specific columns of samples
legend_labels = {
    "OrgB_Control_Vehicle_n1": "Control Vehicle", "OrgB_Control_Treated_n1": "Control Treated", "OrgB_MutA_Vehicle_n1": "MutA Vehicle",  "OrgB_MutA_Treated_n1": "MutA Treated", "OrgB_MutB_Vehicle_n1": "MutB Vehicle",  "OrgB_MutB_Treated_n1": "MutB Treated"
}

# Define legend labels for specific columns of samples1
#legend_labels1 = {
    #"OrgB_Control_Vehicle_n1": "Vehicle", "OrgB_Control_Treated_n1": "Treated"}

# Create the first legend (for samples)
handles1 = [plt.Rectangle((0,0),1,1, color=samples[col]) for col in legend_labels.keys()]
legend1 = plot4.ax_col_dendrogram.legend(handles=handles1, labels=legend_labels.values(),
                                         loc='upper left', bbox_to_anchor=(1.1, -0.75),fontsize='xx-large',  frameon=False)

# Create the second legend (for samples1)
#handles2 = [plt.Rectangle((0,0),1,1, color=samples1[col]) for col in legend_labels1.keys()]
#legend2 = plot4.ax_col_dendrogram.legend(handles=handles2, labels=legend_labels1.values(),
                                        # loc='upper left', bbox_to_anchor=(1.1, -0.45), fontsize='xx-large',  frameon=False)

# Add the legends to the plot
plot4.ax_col_dendrogram.add_artist(legend1)
#plot4.ax_col_dendrogram.add_artist(legend2)


plt.savefig('Clustermap Selected genes only volcano.png', bbox_inches='tight')

plt.show()

In [ ]:
PC3_stat_res3 = DeseqStats(dds1, contrast = ('Condition','MutB-Treated','MutB-Vehicle'))
PC3_stat_res3.summary()
PC3_stat_res3=PC3_stat_res3.results_df
PC3_stat_res3=pd.merge(PC3_stat_res3, data2[['gene_name']], left_index=True, right_index=True)
PC3_stat_res3

In [ ]:
# Copy the downregulated genes data
PC3_stat_res3_rnk = PC3_stat_res3.copy()

# Calculate the rank based on adjusted p-value
PC3_stat_res3_rnk['Rank'] = PC3_stat_res3.stat

# Sort the genes by rank in descending order
PC3_stat_res3_rnk = PC3_stat_res3_rnk.sort_values('Rank', ascending=False)
PC3_stat_res3_rnk

In [ ]:
PC3_res3 = PC3_stat_res3_rnk
PC3_res3

In [ ]:
PC3_MutB_sigs = PC3_res3[(PC3_res3.padj < 0.05)]
PC3_MutB_sigs.to_csv('PC3_MutB_sigs.csv', index=True)
PC3_MutB_sigs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

# Extracting all indices from each set of significant genes
PC3_wgtd_gene1a_data = PC3_Control_sigs.copy()
PC3_wgtd_gene1b_data = PC3_MutA_sigs.copy()
PC3_wgtd_gene1c_data = PC3_MutB_sigs.copy()

# Calculate the total number of unique genes
total_genes_sigs = len(set.union(set(PC3_wgtd_gene1a_data.index), set(PC3_wgtd_gene1b_data.index), set(PC3_wgtd_gene1c_data.index)))

# Create a Venn diagram with equal-sized circles
venn_labels_sigs = {
    '100': len(set(PC3_wgtd_gene1a_data.index) - set(PC3_wgtd_gene1b_data.index) - set(PC3_wgtd_gene1c_data.index)),
    '010': len(set(PC3_wgtd_gene1b_data.index) - set(PC3_wgtd_gene1a_data.index) - set(PC3_wgtd_gene1c_data.index)),
    '001': len(set(PC3_wgtd_gene1c_data.index) - set(PC3_wgtd_gene1a_data.index) - set(PC3_wgtd_gene1b_data.index)),
    '110': len(set(PC3_wgtd_gene1a_data.index) & set(PC3_wgtd_gene1b_data.index) - set(PC3_wgtd_gene1c_data.index)),
    '101': len(set(PC3_wgtd_gene1a_data.index) & set(PC3_wgtd_gene1c_data.index) - set(PC3_wgtd_gene1b_data.index)),
    '011': len(set(PC3_wgtd_gene1b_data.index) & set(PC3_wgtd_gene1c_data.index) - set(PC3_wgtd_gene1a_data.index)),
    '111': len(set(PC3_wgtd_gene1a_data.index) & set(PC3_wgtd_gene1b_data.index) & set(PC3_wgtd_gene1c_data.index))
}

venn_labels_sigs = {key: f"{value}" for key, value in venn_labels_sigs.items()}

# Define custom colors for each set
custom_colors = {
    '100': 'green',   # Control
    '010': 'red',  # MutA
    '001': 'blue',    # MutB
}

# Create a Venn diagram
venn_sigs = venn3([set(PC3_wgtd_gene1a_data.index), set(PC3_wgtd_gene1b_data.index), set(PC3_wgtd_gene1c_data.index)],
                  set_labels=('Control', 'MutA', 'MutB'),
                  normalize_to=total_genes_sigs)

# Set custom colors for each set
for key, value in custom_colors.items():
    venn_sigs.get_patch_by_id(key).set_color(value)

# Set labels for each region in the Venn diagram
for key, value in venn_labels_sigs.items():
    text = venn_sigs.get_label_by_id(key)
    text.set_text(value)
    text.set_fontweight('bold')  # set text to bold

# Set all text in the plot to bold
for text in venn_sigs.set_labels + venn_sigs.subset_labels:
    text.set_fontweight('bold')

# Display the plot
plt.title("Venn Diagram - PC3 weighted Differentially Expressed Genes", y=1.08, fontweight='bold')
plt.savefig('Venn Diagram - PC3 weighted Differentially Expressed Genes.jpg')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming selected_columns is one DataFrame and selected_columns_sigs is another

# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC3_res1['baseMean']), PC3_res1['log2FoldChange'], alpha=0.5, label='Total Genes', color='gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC3_Control_sigs['baseMean']), PC3_Control_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - PC3 weighted Control')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - PC3 weighted Control Total Genes & DEGs.jpg')

# Show the plot
plt.show()


# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC3_res2['baseMean']), PC3_res2['log2FoldChange'], alpha=0.5, label='Total Genes', color='gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC3_MutA_sigs['baseMean']), PC3_MutA_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - PC3 weighted MutA')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - PC3 weighted MutA Total Genes & DEGs.jpg')

# Show the plot
plt.show()


# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC3_res3['baseMean']), PC3_res3['log2FoldChange'], alpha=0.5, label='Total Genes', color='Gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC3_MutB_sigs['baseMean']), PC3_MutB_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - PC3 weighted MutB')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - PC3 weighted MutB Total Genes & DEGs.jpg')

# Show the plot
plt.show()

In [ ]:
# Assuming selected_columns_sigs is your DataFrame

# Create masks for upregulated and downregulated genes for PC3_Control
PC3_Control_upregulated_genes_mask = PC3_Control_sigs['stat'] > 0
PC3_Control_downregulated_genes_mask = PC3_Control_sigs['stat'] < 0

# Extract upregulated and downregulated genes for PC3_Control
PC3_Control_upregulated_genes = PC3_Control_sigs.loc[PC3_Control_upregulated_genes_mask]
PC3_Control_downregulated_genes = PC3_Control_sigs.loc[PC3_Control_downregulated_genes_mask]

# Print the upregulated and downregulated genes for PC3_Control
print("Number of Upregulated Genes - Control:", len(PC3_Control_upregulated_genes))
print("\nControl Upregulated Genes:")
print(PC3_Control_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

# Save upregulated genes to CSV for PC3_Control
PC3_Control_upregulated_genes.to_csv('PC3_Control_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - Control:", len(PC3_Control_downregulated_genes))
print("\nControl Downregulated Genes:")
print(PC3_Control_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

# Save downregulated genes to CSV for PC3_Control
PC3_Control_downregulated_genes.to_csv('PC3_Control_downregulated_genes.csv', index=True)

# Create masks for upregulated and downregulated genes for PC3_MutA
PC3_MutA_upregulated_genes_mask = PC3_MutA_sigs['stat'] > 0
PC3_MutA_downregulated_genes_mask = PC3_MutA_sigs['stat'] < 0

# Extract upregulated and downregulated genes for PC3_MutA
PC3_MutA_upregulated_genes = PC3_MutA_sigs.loc[PC3_MutA_upregulated_genes_mask]
PC3_MutA_downregulated_genes = PC3_MutA_sigs.loc[PC3_MutA_downregulated_genes_mask]

# Print the upregulated and downregulated genes for PC3_MutA
print("Number of Upregulated Genes - MutA:", len(PC3_MutA_upregulated_genes))
print("\nMutA Upregulated Genes:")
print(PC3_MutA_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

# Save upregulated genes to CSV for PC3_MutA
PC3_MutA_upregulated_genes.to_csv('PC3_MutA_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - MutA:", len(PC3_MutA_downregulated_genes))
print("\nMutA Downregulated Genes:")
print(PC3_MutA_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

# Save downregulated genes to CSV for PC3_MutA
PC3_MutA_downregulated_genes.to_csv('PC3_MutA_downregulated_genes.csv', index=True)

# Create masks for upregulated and downregulated genes for PC3_MutB
PC3_MutB_upregulated_genes_mask = PC3_MutB_sigs['stat'] > 0
PC3_MutB_downregulated_genes_mask = PC3_MutB_sigs['stat'] < 0

# Extract upregulated and downregulated genes for PC3_MutB
PC3_MutB_upregulated_genes = PC3_MutB_sigs.loc[PC3_MutB_upregulated_genes_mask]
PC3_MutB_downregulated_genes = PC3_MutB_sigs.loc[PC3_MutB_downregulated_genes_mask]

# Print the upregulated and downregulated genes for PC3_MutB
print("Number of Upregulated Genes - MutB:", len(PC3_MutB_upregulated_genes))
print("\nMutB Upregulated Genes:")
print(PC3_MutB_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

# Save upregulated genes to CSV for PC3_MutB
PC3_MutB_upregulated_genes.to_csv('PC3_MutB_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - MutB:", len(PC3_MutB_downregulated_genes))
print("\nMutB Downregulated Genes:")
print(PC3_MutB_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

# Save downregulated genes to CSV for PC3_MutB
PC3_MutB_downregulated_genes.to_csv('PC3_MutB_downregulated_genes.csv', index=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

# Assuming 'Symbol' column contains gene symbols in your DataFrames

# Extracting all columns from each set of significant genes
PC3_wgtd_geneup1_data = PC3_Control_upregulated_genes.copy()
PC3_wgtd_geneup2_data = PC3_MutA_upregulated_genes.copy()
PC3_wgtd_geneup3_data = PC3_MutB_upregulated_genes.copy()

# Calculate the total number of unique genes
total_genes_sigs = len(set.union(set(PC3_wgtd_geneup1_data.index), set(PC3_wgtd_geneup2_data.index), set(PC3_wgtd_geneup3_data.index)))

# Create a Venn diagram with equal-sized circles
venn_labels_sigs = {'100': len(set(PC3_wgtd_geneup1_data.index) - set(PC3_wgtd_geneup2_data.index) - set(PC3_wgtd_geneup3_data.index)),
                    '010': len(set(PC3_wgtd_geneup2_data.index) - set(PC3_wgtd_geneup1_data.index) - set(PC3_wgtd_geneup3_data.index)),
                    '001': len(set(PC3_wgtd_geneup3_data.index) - set(PC3_wgtd_geneup1_data.index) - set(PC3_wgtd_geneup2_data.index)),
                    '110': len(set(PC3_wgtd_geneup1_data.index) & set(PC3_wgtd_geneup2_data.index) - set(PC3_wgtd_geneup3_data.index)),
                    '101': len(set(PC3_wgtd_geneup1_data.index) & set(PC3_wgtd_geneup3_data.index) - set(PC3_wgtd_geneup2_data.index)),
                    '011': len(set(PC3_wgtd_geneup2_data.index) & set(PC3_wgtd_geneup3_data.index) - set(PC3_wgtd_geneup1_data.index)),
                    '111': len(set(PC3_wgtd_geneup1_data.index) & set(PC3_wgtd_geneup2_data.index) & set(PC3_wgtd_geneup3_data.index))
                   }

venn_labels_sigs = {key: f"{value}" for key, value in venn_labels_sigs.items()}

# Define custom colors for each set
custom_colors = {
    '100': 'green',   # Control
    '010': 'red',  # MutA
    '001': 'blue',    # MutB
}


# Create a DataFrame for each set of genes
PC3_wgtd_Genes_control_upDEGs = PC3_wgtd_geneup1_data.loc[~PC3_wgtd_geneup1_data.index.isin(set(PC3_wgtd_geneup2_data.index) | set(PC3_wgtd_geneup3_data.index))]
PC3_wgtd_Genes_mutA_upDEGs = PC3_wgtd_geneup2_data.loc[~PC3_wgtd_geneup2_data.index.isin(set(PC3_wgtd_geneup1_data.index) | set(PC3_wgtd_geneup3_data.index))]
PC3_wgtd_Genes_mutB_upDEGs = PC3_wgtd_geneup3_data.loc[~PC3_wgtd_geneup3_data.index.isin(set(PC3_wgtd_geneup1_data.index) | set(PC3_wgtd_geneup2_data.index))]
PC3_wgtd_Genes_control_mutA_upDEGs = PC3_wgtd_geneup1_data.loc[PC3_wgtd_geneup1_data.index.isin(set(PC3_wgtd_geneup2_data.index).difference(set(PC3_wgtd_geneup3_data.index)))]
PC3_wgtd_Genes_mutB_mutA_upDEGs = PC3_wgtd_geneup2_data.loc[PC3_wgtd_geneup2_data.index.isin(set(PC3_wgtd_geneup3_data.index).difference(set(PC3_wgtd_geneup1_data.index)))]
PC3_wgtd_Genes_control_mutB_upDEGs = PC3_wgtd_geneup1_data.loc[PC3_wgtd_geneup1_data.index.isin(set(PC3_wgtd_geneup3_data.index).difference(set(PC3_wgtd_geneup2_data.index)))]
PC3_wgtd_Common_Genes_upDEGs = pd.DataFrame({"Genes": list(set(PC3_wgtd_geneup1_data.index) & set(PC3_wgtd_geneup2_data.index) & set(PC3_wgtd_geneup3_data.index))})

# Display or use the extracted genes as needed
print("Genes only expressed in Control:")
print(PC3_wgtd_Genes_control_upDEGs)

print("\nGenes only expressed in MutA:")
print(PC3_wgtd_Genes_mutA_upDEGs)

print("\nGenes only expressed in MutB:")
print(PC3_wgtd_Genes_mutB_upDEGs)

print("\nGenes expressed between Control and MutA:")
print(PC3_wgtd_Genes_control_mutA_upDEGs)

print("\nGenes expressed between MutB and MutA:")
print(PC3_wgtd_Genes_mutB_mutA_upDEGs)

print("\nGenes expressed between Control and MutB:")
print(PC3_wgtd_Genes_control_mutB_upDEGs)

print("\nGenes expressed between Control, MutB, and MutA:")
print(PC3_wgtd_Common_Genes_upDEGs)

# Save genes to CSV
PC3_wgtd_Genes_control_upDEGs.to_csv('PC3_wgtd_Genes_control_upDEGs.csv', index=True)
PC3_wgtd_Genes_mutA_upDEGs.to_csv('PC3_wgtd_Genes_mutA_upDEGs.csv', index=True)
PC3_wgtd_Genes_mutB_upDEGs.to_csv('PC3_wgtd_Genes_mutB_upDEGs.csv', index=True)
PC3_wgtd_Genes_control_mutA_upDEGs.to_csv('PC3_wgtd_Genes_control_mutA_upDEGs.csv', index=True)
PC3_wgtd_Genes_mutB_mutA_upDEGs.to_csv('PC3_wgtd_Genes_mutB_mutA_upDEGs.csv', index=True)
PC3_wgtd_Genes_control_mutB_upDEGs.to_csv('PC3_wgtd_Genes_control_mutB_upDEGs.csv', index=True)
PC3_wgtd_Common_Genes_upDEGs.to_csv('PC3_wgtd_Common_Genes_upDEGs.csv', index=True)

# Create a Venn diagram
venn_DEGs = venn3([set(PC3_wgtd_geneup1_data.index), set(PC3_wgtd_geneup2_data.index), set(PC3_wgtd_geneup3_data.index)],
                  set_labels=('Control', 'MutA', 'MutB'),
                  normalize_to=total_genes_sigs)

# Set custom colors for each set
for key, value in custom_colors.items():
    venn_DEGs.get_patch_by_id(key).set_color(value)

# Set labels for each region in the Venn diagram
for key, value in venn_labels_sigs.items():
    text = venn_DEGs.get_label_by_id(key)
    text.set_text(value)
    text.set_fontweight('bold')  # set text to bold

# Set all text in the plot to bold
for text in venn_DEGs.set_labels + venn_DEGs.subset_labels:
    text.set_fontweight('bold')


# Display the plot
plt.title("Venn Diagram - PC3 weighted Upregulated DEGs", y=1.08, fontweight='bold')
plt.savefig('Venn Diagram - PC3 weighted Upregulated DEGs.jpg')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

# Assuming 'Symbol' column contains gene symbols in your DataFrames

# Extracting all columns from each set of significant genes
PC3_wgtd_genedown1_data = PC3_Control_downregulated_genes.copy()
PC3_wgtd_genedown2_data = PC3_MutA_downregulated_genes.copy()
PC3_wgtd_genedown3_data = PC3_MutB_downregulated_genes.copy()

# Calculate the total number of unique genes
total_genes_sigs = len(set.union(set(PC3_wgtd_genedown1_data.index), set(PC3_wgtd_genedown2_data.index), set(PC3_wgtd_genedown3_data.index)))

# Create a Venn diagram with equal-sized circles
venn_labels_sigs = {'100': len(set(PC3_wgtd_genedown1_data.index) - set(PC3_wgtd_genedown2_data.index) - set(PC3_wgtd_genedown3_data.index)),
                    '010': len(set(PC3_wgtd_genedown2_data.index) - set(PC3_wgtd_genedown1_data.index) - set(PC3_wgtd_genedown3_data.index)),
                    '001': len(set(PC3_wgtd_genedown3_data.index) - set(PC3_wgtd_genedown1_data.index) - set(PC3_wgtd_genedown2_data.index)),
                    '110': len(set(PC3_wgtd_genedown1_data.index) & set(PC3_wgtd_genedown2_data.index) - set(PC3_wgtd_genedown3_data.index)),
                    '101': len(set(PC3_wgtd_genedown1_data.index) & set(PC3_wgtd_genedown3_data.index) - set(PC3_wgtd_genedown2_data.index)),
                    '011': len(set(PC3_wgtd_genedown2_data.index) & set(PC3_wgtd_genedown3_data.index) - set(PC3_wgtd_genedown1_data.index)),
                    '111': len(set(PC3_wgtd_genedown1_data.index) & set(PC3_wgtd_genedown2_data.index) & set(PC3_wgtd_genedown3_data.index))
                   }

venn_labels_sigs = {key: f"{value}" for key, value in venn_labels_sigs.items()}

# Define custom colors for each set
custom_colors = {
    '100': 'green',   # Control
    '010': 'red',  # MutA
    '001': 'blue',    # MutB
}

# Create a DataFrame for each set of genes
PC3_wgtd_Genes_control_downDEGs = PC3_wgtd_genedown1_data.loc[~PC3_wgtd_genedown1_data.index.isin(set(PC3_wgtd_genedown2_data.index) | set(PC3_wgtd_genedown3_data.index))]
PC3_wgtd_Genes_mutA_downDEGs = PC3_wgtd_genedown2_data.loc[~PC3_wgtd_genedown2_data.index.isin(set(PC3_wgtd_genedown1_data.index) | set(PC3_wgtd_genedown3_data.index))]
PC3_wgtd_Genes_mutB_downDEGs = PC3_wgtd_genedown3_data.loc[~PC3_wgtd_genedown3_data.index.isin(set(PC3_wgtd_genedown1_data.index) | set(PC3_wgtd_genedown2_data.index))]
PC3_wgtd_Genes_control_mutA_downDEGs = PC3_wgtd_genedown1_data.loc[PC3_wgtd_genedown1_data.index.isin(set(PC3_wgtd_genedown2_data.index).difference(set(PC3_wgtd_genedown3_data.index)))]
PC3_wgtd_Genes_mutB_mutA_downDEGs = PC3_wgtd_genedown2_data.loc[PC3_wgtd_genedown2_data.index.isin(set(PC3_wgtd_genedown3_data.index).difference(set(PC3_wgtd_genedown1_data.index)))]
PC3_wgtd_Genes_control_mutB_downDEGs = PC3_wgtd_genedown1_data.loc[PC3_wgtd_genedown1_data.index.isin(set(PC3_wgtd_genedown3_data.index).difference(set(PC3_wgtd_genedown2_data.index)))]
PC3_wgtd_Common_Genes_downDEGs = pd.DataFrame({"Genes": list(set(PC3_wgtd_genedown1_data.index) & set(PC3_wgtd_genedown2_data.index) & set(PC3_wgtd_genedown3_data.index))})

# Display or use the extracted genes as needed
print("Genes only expressed in Control:")
print(PC3_wgtd_Genes_control_downDEGs)

print("\nGenes only expressed in MutA:")
print(PC3_wgtd_Genes_mutA_downDEGs)

print("\nGenes only expressed in MutB:")
print(PC3_wgtd_Genes_mutB_downDEGs)

print("\nGenes expressed between Control and MutA:")
print(PC3_wgtd_Genes_control_mutA_downDEGs)

print("\nGenes expressed between MutB and MutA:")
print(PC3_wgtd_Genes_mutB_mutA_downDEGs)

print("\nGenes expressed between Control and MutB:")
print(PC3_wgtd_Genes_control_mutB_downDEGs)

print("\nGenes expressed between Control, MutB, and MutA:")
print(PC3_wgtd_Common_Genes_downDEGs)

# Save genes to CSV
PC3_wgtd_Genes_control_downDEGs.to_csv('PC3_wgtd_Genes_control_downDEGs.csv', index=True)
PC3_wgtd_Genes_mutA_downDEGs.to_csv('PC3_wgtd_Genes_mutA_downDEGs.csv', index=True)
PC3_wgtd_Genes_mutB_downDEGs.to_csv('PC3_wgtd_Genes_mutB_downDEGs.csv', index=True)
PC3_wgtd_Genes_control_mutA_downDEGs.to_csv('PC3_wgtd_Genes_control_mutA_downDEGs.csv', index=True)
PC3_wgtd_Genes_mutB_mutA_downDEGs.to_csv('PC3_wgtd_Genes_mutB_mutA_downDEGs.csv', index=True)
PC3_wgtd_Genes_control_mutB_downDEGs.to_csv('PC3_wgtd_Genes_control_mutB_downDEGs.csv', index=True)
PC3_wgtd_Common_Genes_downDEGs.to_csv('PC3_wgtd_Common_Genes_downDEGs.csv', index=True)

# Create a Venn diagram
venn_DEGs = venn3([set(PC3_wgtd_genedown1_data.index), set(PC3_wgtd_genedown2_data.index), set(PC3_wgtd_genedown3_data.index)],
                  set_labels=('Control', 'MutA', 'MutB'),
                  normalize_to=total_genes_sigs)

# Set custom colors for each set
for key, value in custom_colors.items():
    venn_DEGs.get_patch_by_id(key).set_color(value)

# Set labels for each region in the Venn diagram
for key, value in venn_labels_sigs.items():
    text = venn_DEGs.get_label_by_id(key)
    text.set_text(value)
    text.set_fontweight('bold')  # set text to bold

# Set all text in the plot to bold
for text in venn_DEGs.set_labels + venn_DEGs.subset_labels:
    text.set_fontweight('bold')


# Display the plot
plt.title("Venn Diagram - PC3 weighted Downregulated DEGs", y=1.08, fontweight='bold')
plt.savefig('Venn Diagram - PC3 weighted Downregulated DEGs.jpg')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
PC3_Control_enr_GOBP_up1 = gp.enrichr(
    gene_list=PC3_Control_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_Control_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
PC3_Control_enr_GOBP_down1 = gp.enrichr(
    gene_list=PC3_Control_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_Control_enr_GOBP_down',
    cutoff=0.05
)



# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(PC3_Control_enr_GOBP_up1.results, title="PC3 Control: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('PC3 Control: Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(PC3_Control_enr_GOBP_down1.results, title="PC3 Control: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('PC3 Control: Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results
PC3_Control_enrichment_results_up = PC3_Control_enr_GOBP_up1.results

# Convert the results to a DataFrame
PC3_Control_df_upregulated = pd.DataFrame(PC3_Control_enrichment_results_up)

# Save the DataFrame to a CSV file
PC3_Control_df_upregulated.to_csv('PC3_Control_df_upregulated.csv', index=False)

import pandas as pd

# Extracting relevant information from the results
PC3_Control_enrichment_results_down = PC3_Control_enr_GOBP_down1.results

# Convert the results to a DataFrame
PC3_Control_df_downregulated = pd.DataFrame(PC3_Control_enrichment_results_down)

# Save the DataFrame to a CSV file
PC3_Control_df_downregulated.to_csv('PC3_Control_df_downregulated.csv', index=False)


In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_Control_enrchd_up_genes = []

# Get the number of rows in the DataFrame
Control_up_num_rows = len(PC3_Control_df_upregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, Control_up_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    Control_up_lead_genes_value = PC3_Control_df_upregulated['Genes'].iloc[i]
    PC3_Control_up_enrchd_genes = Control_up_lead_genes_value.split(";")
    PC3_Control_enrchd_up_genes.extend(PC3_Control_up_enrchd_genes)

# Remove duplicates from the list of genes
PC3_Control_enrchd_up_unique_genes = list(set(PC3_Control_enrchd_up_genes))

# Create a new DataFrame with unique genes
PC3_Control_up_genes_df = pd.DataFrame(PC3_Control_enrchd_up_unique_genes, columns=['Genes'])
PC3_Control_up_genes_df.to_csv('PC3_Control_up_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_Control_up_genes_df)

import pandas as pd

# Initialize an empty list to store genes
PC3_Control_enrchd_down_genes = []

# Get the number of rows in the DataFrame
Control_down_num_rows = len(PC3_Control_df_downregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, Control_down_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    Control_down_lead_genes_value = PC3_Control_df_downregulated['Genes'].iloc[i]
    PC3_Control_down_enrchd_genes = Control_down_lead_genes_value.split(";")
    PC3_Control_enrchd_down_genes.extend(PC3_Control_down_enrchd_genes)

# Remove duplicates from the list of genes
PC3_Control_enrchd_down_unique_genes = list(set(PC3_Control_enrchd_down_genes))

# Create a new DataFrame with unique genes
PC3_Control_down_genes_df = pd.DataFrame(PC3_Control_enrchd_down_unique_genes, columns=['Genes'])
PC3_Control_down_genes_df.to_csv('PC3_Control_down_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_Control_down_genes_df)

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
PC3_Control_only_enr_GOBP_up1 = gp.enrichr(
    gene_list=PC3_wgtd_Genes_control_upDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_Control_only_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
PC3_Control_only_enr_GOBP_down1 = gp.enrichr(
    gene_list=PC3_wgtd_Genes_control_downDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_Control_only_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(PC3_Control_only_enr_GOBP_up1.results, title="PC3 Control_only: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('PC3 Control_only: Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(PC3_Control_only_enr_GOBP_down1.results, title="PC3 Control_only: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('PC3 Control_only: Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results
PC3_Control_only_enrichment_results_up = PC3_Control_only_enr_GOBP_up1.results

# Convert the results to a DataFrame
PC3_Control_only_df_upregulated = pd.DataFrame(PC3_Control_only_enrichment_results_up)

# Save the DataFrame to a CSV file
PC3_Control_only_df_upregulated.to_csv('PC3_Control_only_df_upregulated.csv', index=False)

import pandas as pd

# Extracting relevant information from the results
PC3_Control_only_enrichment_results_down = PC3_Control_only_enr_GOBP_down1.results

# Convert the results to a DataFrame
PC3_Control_only_df_downregulated = pd.DataFrame(PC3_Control_only_enrichment_results_down)

# Save the DataFrame to a CSV file
PC3_Control_only_df_downregulated.to_csv('PC3_Control_only_df_downregulated.csv', index=False)


In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_Control_only_enrchd_up_genes = []

# Get the number of rows in the DataFrame
Control_only_up_num_rows = len(PC3_Control_only_df_upregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, Control_only_up_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    Control_only_up_lead_genes_value = PC3_Control_only_df_upregulated['Genes'].iloc[i]
    PC3_Control_only_up_enrchd_genes = Control_only_up_lead_genes_value.split(";")
    PC3_Control_only_enrchd_up_genes.extend(PC3_Control_only_up_enrchd_genes)

# Remove duplicates from the list of genes
PC3_Control_only_enrchd_up_unique_genes = list(set(PC3_Control_only_enrchd_up_genes))

# Create a new DataFrame with unique genes
PC3_Control_only_up_genes_df = pd.DataFrame(PC3_Control_only_enrchd_up_unique_genes, columns=['Genes'])
PC3_Control_only_up_genes_df.to_csv('PC3_Control_only_up_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_Control_only_up_genes_df)

import pandas as pd

# Initialize an empty list to store genes
PC3_Control_only_enrchd_down_genes = []

# Get the number of rows in the DataFrame
Control_only_down_num_rows = len(PC3_Control_only_df_downregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, Control_only_down_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    Control_only_down_lead_genes_value = PC3_Control_only_df_downregulated['Genes'].iloc[i]
    PC3_Control_only_down_enrchd_genes = Control_only_down_lead_genes_value.split(";")
    PC3_Control_only_enrchd_down_genes.extend(PC3_Control_only_down_enrchd_genes)

# Remove duplicates from the list of genes
PC3_Control_only_enrchd_down_unique_genes = list(set(PC3_Control_only_enrchd_down_genes))

# Create a new DataFrame with unique genes
PC3_Control_only_down_genes_df = pd.DataFrame(PC3_Control_only_enrchd_down_unique_genes, columns=['Genes'])
PC3_Control_only_down_genes_df.to_csv('PC3_Control_only_down_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_Control_only_down_genes_df)

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
PC3_MutA_enr_GOBP_up1 = gp.enrichr(
    gene_list=PC3_MutA_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutA_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
PC3_MutA_enr_GOBP_down1 = gp.enrichr(
    gene_list=PC3_MutA_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutA_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(PC3_MutA_enr_GOBP_up1.results, title="PC3 MutA: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('PC3 MutA: Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(PC3_MutA_enr_GOBP_down1.results, title="PC3 MutA: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('PC3 MutA: Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results
PC3_MutA_enrichment_results_up = PC3_MutA_enr_GOBP_up1.results

# Convert the results to a DataFrame
PC3_MutA_df_upregulated = pd.DataFrame(PC3_MutA_enrichment_results_up)

# Save the DataFrame to a CSV file
PC3_MutA_df_upregulated.to_csv('PC3_MutA_df_upregulated.csv', index=False)

import pandas as pd

# Extracting relevant information from the results
PC3_MutA_enrichment_results_down = PC3_MutA_enr_GOBP_down1.results

# Convert the results to a DataFrame
PC3_MutA_df_downregulated = pd.DataFrame(PC3_MutA_enrichment_results_down)

# Save the DataFrame to a CSV file
PC3_MutA_df_downregulated.to_csv('PC3_MutA_df_downregulated.csv', index=False)


In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_MutA_enrchd_up_genes = []

# Get the number of rows in the DataFrame
MutA_up_num_rows = len(PC3_MutA_df_upregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutA_up_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutA_up_lead_genes_value = PC3_MutA_df_upregulated['Genes'].iloc[i]
    PC3_MutA_up_enrchd_genes = MutA_up_lead_genes_value.split(";")
    PC3_MutA_enrchd_up_genes.extend(PC3_MutA_up_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutA_enrchd_up_unique_genes = list(set(PC3_MutA_enrchd_up_genes))

# Create a new DataFrame with unique genes
PC3_MutA_up_genes_df = pd.DataFrame(PC3_MutA_enrchd_up_unique_genes, columns=['Genes'])
PC3_MutA_up_genes_df.to_csv('PC3_MutA_up_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutA_up_genes_df)

import pandas as pd

# Initialize an empty list to store genes
PC3_MutA_enrchd_down_genes = []

# Get the number of rows in the DataFrame
MutA_down_num_rows = len(PC3_MutA_df_downregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutA_down_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutA_down_lead_genes_value = PC3_MutA_df_downregulated['Genes'].iloc[i]
    PC3_MutA_down_enrchd_genes = MutA_down_lead_genes_value.split(";")
    PC3_MutA_enrchd_down_genes.extend(PC3_MutA_down_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutA_enrchd_down_unique_genes = list(set(PC3_MutA_enrchd_down_genes))

# Create a new DataFrame with unique genes
PC3_MutA_down_genes_df = pd.DataFrame(PC3_MutA_enrchd_down_unique_genes, columns=['Genes'])
PC3_MutA_down_genes_df.to_csv('PC3_MutA_down_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutA_down_genes_df)


In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
PC3_MutA_only_enr_GOBP_up1 = gp.enrichr(
    gene_list=PC3_wgtd_Genes_mutA_upDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutA_only_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
PC3_MutA_only_enr_GOBP_down1 = gp.enrichr(
    gene_list=PC3_wgtd_Genes_mutA_downDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutA_only_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(PC3_MutA_only_enr_GOBP_up1.results, title="PC3 MutA_only: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('PC3 MutA_only: Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(PC3_MutA_only_enr_GOBP_down1.results, title="PC3 MutA_only: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('PC3 MutA_only: Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
# Assuming MutA_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
MutA_up_rnk = PC3_wgtd_Genes_mutA_upDEGs.copy()

# Sort the genes by rank in descending order
MutA_up_rnk = MutA_up_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
MutA_up_ranking = MutA_up_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
MutA_up_ranking.to_csv('MutA_up_ranking.csv', index=False)

# Perform gene set enrichment analysis
MutA_up_pre_res = gp.prerank(rnk=MutA_up_ranking, gene_sets='GO_Biological_Process_2023',
                                seed=6, permutation_num=100, min_size=8, max_size=1000)

# Filter out terms with FDR q-value equal to zero
MutA_up_pre_res.res2d = MutA_up_pre_res.res2d[MutA_up_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
MutA_up_pre_res.res2d = MutA_up_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
MutA_up_terms = MutA_up_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
MutA_up_axs = MutA_up_pre_res.plot(terms=MutA_up_terms,
                                         show_ranking=True,
                                         figsize=(3, 4))

# Save the plot
plt.savefig('MutA_up_rank.jpg', bbox_inches="tight")

MutA_up_pre_res.res2d.to_csv('MutA_up_pre_res.csv')
MutA_up_pre_res.res2d


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from gseapy import enrichment_map
import textwrap

# Load data and create the graph
MutA_up_nodes, MutA_up_edges = enrichment_map(MutA_up_pre_res.res2d, cutoff=1)
MutA_up_G = nx.from_pandas_edgelist(MutA_up_edges, source='src_idx', target='targ_idx', edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes'])

# Initialize the figure and axis
fig, ax = plt.subplots(figsize=(12, 15))

# Initialize node positions using Kamada-Kawai layout
MutA_up_pos = nx.layout.kamada_kawai_layout(MutA_up_G)

# Function to wrap text for node labels
def wrap_text(text, width=15):
    wrapped_text = textwrap.fill(text, width)
    return wrapped_text

# Node color
MutA_up_node_color = 'IndianRed'  # Set all nodes to red

# Draw nodes with updated color and size
nx.draw_networkx_nodes(MutA_up_G,
                       pos=MutA_up_pos,
                       node_color=MutA_up_node_color,
                       node_size=MutA_up_nodes.Hits_ratio * 5000,
                       alpha=0.7)

# Draw node labels with improved positioning, font size, and wrapped text
wrapped_labels = {node: wrap_text(label) for node, label in MutA_up_nodes.Term.to_dict().items()}
nx.draw_networkx_labels(MutA_up_G,
                        pos=MutA_up_pos,
                        labels=wrapped_labels,
                        font_size=15,
                        font_weight='bold',  # Adding bold font
                        verticalalignment='bottom')

# Draw edges with adjusted color, width, and weight
MutA_up_edge_weight = nx.get_edge_attributes(MutA_up_G, 'jaccard_coef').values()
MutA_up_edge_width = [w * 5 for w in MutA_up_edge_weight]
MutA_up_edge_color = MutA_up_edge_weight  # Using the same color for edges as weight

# Color mapping for edges
MutA_up_edge_cmap = plt.cm.RdYlBu  # Choosing colormap for edge colors
MutA_up_edge_norm = plt.Normalize(min(MutA_up_edge_weight), max(MutA_up_edge_weight))  # Normalizing edge weights for colormap

nx.draw_networkx_edges(MutA_up_G,
                       pos=MutA_up_pos,
                       width=MutA_up_edge_width,
                       edge_color=MutA_up_edge_weight,
                       edge_cmap=MutA_up_edge_cmap,
                       edge_vmin=min(MutA_up_edge_weight),
                       edge_vmax=max(MutA_up_edge_weight),
                       alpha=0.5)

# Colorbar for edge color mapping
#MutA_up_edge_cbar = plt.colorbar(plt.cm.ScalarMappable(norm=MutA_up_edge_norm, cmap=MutA_up_edge_cmap), ax=ax, shrink=0.25, pad=0.05, anchor=(-1.25, -0.000001))
#MutA_up_edge_cbar.set_label('Edges')

plt.tight_layout()

plt.savefig('MutA_up_network_map.jpg', dpi=300)

# Show the plot
plt.show()


In [ ]:
# Assuming MutA_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the downregulated genes data
MutA_down_rnk = PC3_wgtd_Genes_mutA_downDEGs.copy()

# Sort the genes by rank in descending order
MutA_down_rnk = MutA_down_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
MutA_down_ranking = MutA_down_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
MutA_down_ranking.to_csv('MutA_down_ranking.csv', index=False)

# Perform gene set enrichment analysis
MutA_down_pre_res = gp.prerank(rnk=MutA_down_ranking, gene_sets='GO_Biological_Process_2023',
                                seed=6, permutation_num=100, min_size=2, max_size=100)

# Filter out terms with FDR q-value equal to zero
MutA_down_pre_res.res2d = MutA_down_pre_res.res2d[MutA_down_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
MutA_down_pre_res.res2d = MutA_down_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
MutA_down_terms = MutA_down_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
MutA_down_axs = MutA_down_pre_res.plot(terms=MutA_down_terms,
                                         show_ranking=True,
                                         figsize=(3, 4))

# Save the plot
plt.savefig('MutA_down_rank.jpg', bbox_inches="tight")

MutA_down_pre_res.res2d.to_csv('MutA_down_pre_res.csv')
MutA_down_pre_res.res2d


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from gseapy import enrichment_map
import textwrap

# Load data and create the graph
MutA_down_nodes, MutA_down_edges = enrichment_map(MutA_down_pre_res.res2d, cutoff=5)
MutA_down_G = nx.from_pandas_edgelist(MutA_down_edges, source='src_idx', target='targ_idx', edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes'])

# Initialize the figure and axis
fig, ax = plt.subplots(figsize=(12, 15))

# Initialize node positions using Kamada-Kawai layout
MutA_down_pos = nx.layout.kamada_kawai_layout(MutA_down_G)

# Function to wrap text for node labels
def wrap_text(text, width=15):
    wrapped_text = textwrap.fill(text, width)
    return wrapped_text

# Node color
MutA_down_node_color = 'RoyalBlue'  # Set all nodes to blue

# Draw nodes with updated color and size
nx.draw_networkx_nodes(MutA_down_G,
                       pos=MutA_down_pos,
                       node_color=MutA_down_node_color,
                       node_size=MutA_down_nodes.Hits_ratio * 5000,
                       alpha=0.7)

# Draw node labels with improved positioning, font size, and wrapped text
wrapped_labels = {node: wrap_text(label) for node, label in MutA_down_nodes.Term.to_dict().items()}
nx.draw_networkx_labels(MutA_down_G,
                        pos=MutA_down_pos,
                        labels=wrapped_labels,
                        font_size=15,
                        font_weight='bold',  # Adding bold font
                        verticalalignment='bottom')

# Draw edges with adjusted color, width, and weight
MutA_down_edge_weight = nx.get_edge_attributes(MutA_down_G, 'jaccard_coef').values()
MutA_down_edge_width = [w * 5 for w in MutA_down_edge_weight]
MutA_down_edge_color = MutA_down_edge_weight  # Using the same color for edges as weight

# Color mapping for edges
MutA_down_edge_cmap = plt.cm.RdYlBu  # Choosing colormap for edge colors
MutA_down_edge_norm = plt.Normalize(min(MutA_down_edge_weight), max(MutA_down_edge_weight))  # Normalizing edge weights for colormap

nx.draw_networkx_edges(MutA_down_G,
                       pos=MutA_down_pos,
                       width=MutA_down_edge_width,
                       edge_color=MutA_down_edge_weight,
                       edge_cmap=MutA_down_edge_cmap,
                       edge_vmin=min(MutA_down_edge_weight),
                       edge_vmax=max(MutA_down_edge_weight),
                       alpha=0.5)

# Colorbar for edge color mapping
#MutA_down_edge_cbar = plt.colorbar(plt.cm.ScalarMappable(norm=MutA_down_edge_norm, cmap=MutA_down_edge_cmap), ax=ax, shrink=0.25, pad=0.05, anchor=(-1.25, -0.000001))
#MutA_down_edge_cbar.set_label('Edges')

plt.tight_layout()

plt.savefig('MutA_down_network_map.jpg', dpi=300)

# Show the plot
plt.show()


In [ ]:
import pandas as pd

# Extracting relevant information from the results
PC3_MutA_only_enrichment_results_up = PC3_MutA_only_enr_GOBP_up1.results

# Convert the results to a DataFrame
PC3_MutA_only_df_upregulated = pd.DataFrame(PC3_MutA_only_enrichment_results_up)

# Save the DataFrame to a CSV file
PC3_MutA_only_df_upregulated.to_csv('PC3_MutA_only_df_upregulated.csv', index=False)

import pandas as pd

# Extracting relevant information from the results
PC3_MutA_only_enrichment_results_down = PC3_MutA_only_enr_GOBP_down1.results

# Convert the results to a DataFrame
PC3_MutA_only_df_downregulated = pd.DataFrame(PC3_MutA_only_enrichment_results_down)

# Save the DataFrame to a CSV file
PC3_MutA_only_df_downregulated.to_csv('PC3_MutA_only_df_downregulated.csv', index=False)

In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_MutA_only_enrchd_up_genes = []

# Get the number of rows in the DataFrame
MutA_only_up_num_rows = len(PC3_MutA_only_df_upregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutA_only_up_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutA_only_up_lead_genes_value = PC3_MutA_only_df_upregulated['Genes'].iloc[i]
    PC3_MutA_only_up_enrchd_genes = MutA_only_up_lead_genes_value.split(";")
    PC3_MutA_only_enrchd_up_genes.extend(PC3_MutA_only_up_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutA_only_enrchd_up_unique_genes = list(set(PC3_MutA_only_enrchd_up_genes))

# Create a new DataFrame with unique genes
PC3_MutA_only_up_genes_df = pd.DataFrame(PC3_MutA_only_enrchd_up_unique_genes, columns=['Genes'])
PC3_MutA_only_up_genes_df.to_csv('PC3_MutA_only_up_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutA_only_up_genes_df)

import pandas as pd

# Initialize an empty list to store genes
PC3_MutA_only_enrchd_down_genes = []

# Get the number of rows in the DataFrame
MutA_only_down_num_rows = len(PC3_MutA_only_df_downregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutA_only_down_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutA_only_down_lead_genes_value = PC3_MutA_only_df_downregulated['Genes'].iloc[i]
    PC3_MutA_only_down_enrchd_genes = MutA_only_down_lead_genes_value.split(";")
    PC3_MutA_only_enrchd_down_genes.extend(PC3_MutA_only_down_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutA_only_enrchd_down_unique_genes = list(set(PC3_MutA_only_enrchd_down_genes))

# Create a new DataFrame with unique genes
PC3_MutA_only_down_genes_df = pd.DataFrame(PC3_MutA_only_enrchd_down_unique_genes, columns=['Genes'])
PC3_MutA_only_down_genes_df.to_csv('PC3_MutA_only_down_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutA_only_down_genes_df)


In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
PC3_MutB_enr_GOBP_up1 = gp.enrichr(
    gene_list=PC3_MutB_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutB_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
PC3_MutB_enr_GOBP_down1 = gp.enrichr(
    gene_list=PC3_MutB_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutB_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(PC3_MutB_enr_GOBP_up1.results, title="PC3 MutB: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('PC3 MutB: Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(PC3_MutB_enr_GOBP_down1.results, title="PC3 MutB: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('PC3 MutB: Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Extracting relevant information from the results
PC3_MutB_enrichment_results_up = PC3_MutB_enr_GOBP_up1.results

# Convert the results to a DataFrame
PC3_MutB_df_upregulated = pd.DataFrame(PC3_MutB_enrichment_results_up)

# Save the DataFrame to a CSV file
PC3_MutB_df_upregulated.to_csv('PC3_MutB_df_upregulated.csv', index=False)

import pandas as pd

# Extracting relevant information from the results
PC3_MutB_enrichment_results_down = PC3_MutB_enr_GOBP_down1.results

# Convert the results to a DataFrame
PC3_MutB_df_downregulated = pd.DataFrame(PC3_MutB_enrichment_results_down)

# Save the DataFrame to a CSV file
PC3_MutB_df_downregulated.to_csv('PC3_MutB_df_downregulated.csv', index=False)


In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_MutB_enrchd_up_genes = []

# Get the number of rows in the DataFrame
MutB_up_num_rows = len(PC3_MutB_df_upregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutB_up_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutB_up_lead_genes_value = PC3_MutB_df_upregulated['Genes'].iloc[i]
    PC3_MutB_up_enrchd_genes = MutB_up_lead_genes_value.split(";")
    PC3_MutB_enrchd_up_genes.extend(PC3_MutB_up_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutB_enrchd_up_unique_genes = list(set(PC3_MutB_enrchd_up_genes))

# Create a new DataFrame with unique genes
PC3_MutB_up_genes_df = pd.DataFrame(PC3_MutB_enrchd_up_unique_genes, columns=['Genes'])
PC3_MutB_up_genes_df.to_csv('PC3_MutB_up_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutB_up_genes_df)

import pandas as pd

# Initialize an empty list to store genes
PC3_MutB_enrchd_down_genes = []

# Get the number of rows in the DataFrame
MutB_down_num_rows = len(PC3_MutB_df_downregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutB_down_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutB_down_lead_genes_value = PC3_MutB_df_downregulated['Genes'].iloc[i]
    PC3_MutB_down_enrchd_genes = MutB_down_lead_genes_value.split(";")
    PC3_MutB_enrchd_down_genes.extend(PC3_MutB_down_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutB_enrchd_down_unique_genes = list(set(PC3_MutB_enrchd_down_genes))

# Create a new DataFrame with unique genes
PC3_MutB_down_genes_df = pd.DataFrame(PC3_MutB_enrchd_down_unique_genes, columns=['Genes'])
PC3_MutB_down_genes_df.to_csv('PC3_MutB_down_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutB_down_genes_df)


In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
PC3_MutB_only_enr_GOBP_up1 = gp.enrichr(
    gene_list=PC3_wgtd_Genes_mutB_upDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutB_only_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
PC3_MutB_only_enr_GOBP_down1 = gp.enrichr(
    gene_list=PC3_wgtd_Genes_mutB_downDEGs['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC3_MutB_only_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(PC3_MutB_only_enr_GOBP_up1.results, title="PC3 MutB_only: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('PC3 MutB_only: Enriched GOBP (Upregulated Genes).jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(PC3_MutB_only_enr_GOBP_down1.results, title="PC3 MutB_only: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('PC3 MutB_only: Enriched GOBP (Downregulated Genes).jpg', bbox_inches="tight")

plt.show()


In [ ]:
import pandas as pd

# Extracting relevant information from the results
PC3_MutB_only_enrichment_results_up = PC3_MutB_only_enr_GOBP_up1.results

# Convert the results to a DataFrame
PC3_MutB_only_df_upregulated = pd.DataFrame(PC3_MutB_only_enrichment_results_up)

# Save the DataFrame to a CSV file
PC3_MutB_only_df_upregulated.to_csv('PC3_MutB_only_df_upregulated.csv', index=False)

import pandas as pd

# Extracting relevant information from the results
PC3_MutB_only_enrichment_results_down = PC3_MutB_only_enr_GOBP_down1.results

# Convert the results to a DataFrame
PC3_MutB_only_df_downregulated = pd.DataFrame(PC3_MutB_only_enrichment_results_down)

# Save the DataFrame to a CSV file
PC3_MutB_only_df_downregulated.to_csv('PC3_MutB_only_df_downregulated.csv', index=False)


In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_MutB_only_enrchd_up_genes = []

# Get the number of rows in the DataFrame
MutB_only_up_num_rows = len(PC3_MutB_only_df_upregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutB_only_up_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutB_only_up_lead_genes_value = PC3_MutB_only_df_upregulated['Genes'].iloc[i]
    PC3_MutB_only_up_enrchd_genes = MutB_only_up_lead_genes_value.split(";")
    PC3_MutB_only_enrchd_up_genes.extend(PC3_MutB_only_up_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutB_only_enrchd_up_unique_genes = list(set(PC3_MutB_only_enrchd_up_genes))

# Create a new DataFrame with unique genes
PC3_MutB_only_up_genes_df = pd.DataFrame(PC3_MutB_only_enrchd_up_unique_genes, columns=['Genes'])
PC3_MutB_only_up_genes_df.to_csv('PC3_MutB_only_up_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutB_only_up_genes_df)

import pandas as pd

# Initialize an empty list to store genes
PC3_MutB_only_enrchd_down_genes = []

# Get the number of rows in the DataFrame
MutB_only_down_num_rows = len(PC3_MutB_only_df_downregulated['Genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, MutB_only_down_num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    MutB_only_down_lead_genes_value = PC3_MutB_only_df_downregulated['Genes'].iloc[i]
    PC3_MutB_only_down_enrchd_genes = MutB_only_down_lead_genes_value.split(";")
    PC3_MutB_only_enrchd_down_genes.extend(PC3_MutB_only_down_enrchd_genes)

# Remove duplicates from the list of genes
PC3_MutB_only_enrchd_down_unique_genes = list(set(PC3_MutB_only_enrchd_down_genes))

# Create a new DataFrame with unique genes
PC3_MutB_only_down_genes_df = pd.DataFrame(PC3_MutB_only_enrchd_down_unique_genes, columns=['Genes'])
PC3_MutB_only_down_genes_df.to_csv('PC3_MutB_only_down_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_MutB_only_down_genes_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gseapy as gp

# Assuming PC3_Control_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
PC3_wgtd_Control_rnk = PC3_stat_res1_rnk.copy()

# Sort the genes by rank in descending order
PC3_wgtd_Control_rnk = PC3_wgtd_Control_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
PC3_wgtd_Control_ranking = PC3_wgtd_Control_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
PC3_wgtd_Control_ranking.to_csv('PC3_wgtd_Control_ranking.csv', index=False)

# Perform gene set enrichment analysis
PC3_wgtd_Control_pre_res = gp.prerank(rnk=PC3_wgtd_Control_ranking, gene_sets='GO_Biological_Process_2023',
                                     seed=6, permutation_num=100, min_size=20, max_size=1000)

# Filter out terms with FDR q-value equal to zero
PC3_wgtd_Control_pre_res.res2d = PC3_wgtd_Control_pre_res.res2d[PC3_wgtd_Control_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
PC3_wgtd_Control_pre_res.res2d = PC3_wgtd_Control_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
PC3_wgtd_Control_terms = PC3_wgtd_Control_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
PC3_wgtd_Control_axs = PC3_wgtd_Control_pre_res.plot(terms=PC3_wgtd_Control_terms,
                                              show_ranking=True,
                                              figsize=(3, 4))

# Save the plot
plt.savefig('PC3_wgtd_Control_rank.jpg', bbox_inches="tight")

PC3_wgtd_Control_pre_res.res2d.to_csv('PC3_wgtd_Control_pre_res.csv')
PC3_wgtd_Control_pre_res.res2d

In [ ]:
# Filter the DataFrame
PC3_wgtd_Control_pre_res_df = PC3_wgtd_Control_pre_res.res2d[PC3_wgtd_Control_pre_res.res2d['FDR q-val'] < 0.05]
PC3_wgtd_Control_pre_res_df.head(100000)

In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_wgtd_Control_all_genes = []

# Get the number of rows in the DataFrame
num_rows = len(PC3_wgtd_Control_pre_res_df['Lead_genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    lead_genes_value = PC3_wgtd_Control_pre_res_df['Lead_genes'].iloc[i]
    PC3_wgtd_Control_genes = lead_genes_value.split(";")
    PC3_wgtd_Control_all_genes.extend(PC3_wgtd_Control_genes)

# Remove duplicates from the list of genes
PC3_wgtd_Control_unique_genes = list(set(PC3_wgtd_Control_all_genes))

# Create a new DataFrame with unique genes
PC3_wgtd_Control_unique_genes_df = pd.DataFrame(PC3_wgtd_Control_unique_genes, columns=['Genes'])
PC3_wgtd_Control_unique_genes_df.to_csv('PC3_wgtd_Control_unique_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_wgtd_Control_unique_genes_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gseapy as gp

# Assuming PC3_MutA_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
PC3_wgtd_MutA_rnk = PC3_stat_res2_rnk.copy()

# Sort the genes by rank in descending order
PC3_wgtd_MutA_rnk = PC3_wgtd_MutA_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
PC3_wgtd_MutA_ranking = PC3_wgtd_MutA_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
PC3_wgtd_MutA_ranking.to_csv('PC3_wgtd_MutA_ranking.csv', index=False)

# Perform gene set enrichment analysis
PC3_wgtd_MutA_pre_res = gp.prerank(rnk=PC3_wgtd_MutA_ranking, gene_sets='GO_Biological_Process_2023',
                                      seed=6, permutation_num=100, min_size=20, max_size=1000)

# Filter out terms with FDR q-value equal to zero
PC3_wgtd_MutA_pre_res.res2d = PC3_wgtd_MutA_pre_res.res2d[PC3_wgtd_MutA_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
PC3_wgtd_MutA_pre_res.res2d = PC3_wgtd_MutA_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
PC3_wgtd_MutA_terms = PC3_wgtd_MutA_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
PC3_wgtd_MutA_axs = PC3_wgtd_MutA_pre_res.plot(terms=PC3_wgtd_MutA_terms,
                                                     show_ranking=True,
                                                     figsize=(3, 4))

# Save the plot
plt.savefig('PC3_wgtd_MutA_rank.jpg', bbox_inches="tight")

PC3_wgtd_MutA_pre_res.res2d.to_csv('PC3_wgtd_MutA_pre_res.csv')
PC3_wgtd_MutA_pre_res.res2d


In [ ]:
PC3_MutA_sigs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gseapy as gp

# Assuming PC3_MutA_Sigs_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
PC3_wgtd_MutA_Sigs_rnk = PC3_stat_res2_rnk.copy()

# Sort the genes by rank in descending order
PC3_wgtd_MutA_Sigs_rnk = PC3_wgtd_MutA_Sigs_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
PC3_wgtd_MutA_Sigs_ranking = PC3_wgtd_MutA_Sigs_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
PC3_wgtd_MutA_Sigs_ranking.to_csv('PC3_wgtd_MutA_Sigs_ranking.csv', index=False)

# Perform gene set enrichment analysis
PC3_wgtd_MutA_Sigs_pre_res = gp.prerank(rnk=PC3_wgtd_MutA_Sigs_ranking, gene_sets='GO_Biological_Process_2023',
                                           seed=6, permutation_num=100, min_size=100, max_size=1000)

# Filter out terms with FDR q-value equal to zero
PC3_wgtd_MutA_Sigs_pre_res.res2d = PC3_wgtd_MutA_Sigs_pre_res.res2d[PC3_wgtd_MutA_Sigs_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
PC3_wgtd_MutA_Sigs_pre_res.res2d = PC3_wgtd_MutA_Sigs_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
PC3_wgtd_MutA_Sigs_terms = PC3_wgtd_MutA_Sigs_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
PC3_wgtd_MutA_Sigs_axs = PC3_wgtd_MutA_Sigs_pre_res.plot(terms=PC3_wgtd_MutA_Sigs_terms,
                                                               show_ranking=True,
                                                               figsize=(3, 4))

# Save the plot
plt.savefig('PC3_wgtd_MutA_Sigs_rank.jpg', bbox_inches="tight")

# Save the results to a CSV file
PC3_wgtd_MutA_Sigs_pre_res.res2d.to_csv('PC3_wgtd_MutA_Sigs_pre_res.csv')

# Display the results
PC3_wgtd_MutA_Sigs_pre_res.res2d


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from gseapy import enrichment_map
import textwrap

# Load data and create the graph
MutA_nodes, MutA_edges = enrichment_map(PC3_wgtd_MutA_Sigs_pre_res.res2d, cutoff=5)
MutA_G = nx.from_pandas_edgelist(MutA_edges, source='src_idx', target='targ_idx', edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes'])

# Initialize the figure and axis
fig, ax = plt.subplots(figsize=(17, 15))

# Initialize node positions using Kamada-Kawai layout
MutA_pos = nx.layout.kamada_kawai_layout(MutA_G)

# Function to wrap text for node labels
def wrap_text(text, width=15):
    wrapped_text = textwrap.fill(text, width)
    return wrapped_text

# Node color map
MutA_node_color = MutA_nodes.NES
MutA_cmap = plt.cm.RdYlBu
MutA_norm = plt.Normalize(MutA_node_color.min(), MutA_node_color.max())

# Check the sizes of node attributes
print("Number of nodes in graph:", len(MutA_G.nodes()))
print("Size of MutA_nodes DataFrame:", len(MutA_nodes))
print("Size of node_color array:", len(MutA_node_color))
print("Size of node_size array:", len(MutA_nodes.Hits_ratio))

# Convert Hits_ratio to numpy array and ensure it matches the number of nodes
MutA_node_size = (MutA_nodes.Hits_ratio * 5000).values
print("Converted size of node_size array:", len(MutA_node_size))

# Draw nodes with updated color map and size
nx.draw_networkx_nodes(MutA_G,
                       pos=MutA_pos,
                       cmap=MutA_cmap,
                       node_color=MutA_node_color,
                       node_size=MutA_node_size,  # Ensure this is a numpy array
                       alpha=0.7,
                       vmin=MutA_node_color.min(),
                       vmax=MutA_node_color.max())

# Draw node labels with improved positioning, font size, and wrapped text
wrapped_labels = {node: wrap_text(label) for node, label in MutA_nodes.Term.to_dict().items()}
nx.draw_networkx_labels(MutA_G,
                        pos=MutA_pos,
                        labels=wrapped_labels,
                        font_size=15,
                        font_weight='bold',  # Adding bold font
                        verticalalignment='bottom')

# Draw edges with adjusted color, width, and weight
MutA_edge_weight = nx.get_edge_attributes(MutA_G, 'jaccard_coef').values()
MutA_edge_width = [w * 5 for w in MutA_edge_weight]
MutA_edge_color = MutA_edge_weight  # Using the same color for edges as weight

# Color mapping for edges
MutA_edge_cmap = plt.cm.RdYlBu  # Choosing colormap for edge colors
MutA_edge_norm = plt.Normalize(min(MutA_edge_weight), max(MutA_edge_weight))  # Normalizing edge weights for colormap

nx.draw_networkx_edges(MutA_G,
                       pos=MutA_pos,
                       width=MutA_edge_width,
                       edge_color=MutA_edge_weight,
                       edge_cmap=MutA_edge_cmap,
                       edge_vmin=min(MutA_edge_weight),
                       edge_vmax=max(MutA_edge_weight),
                       alpha=0.5)

# Colorbar for edge color mapping
MutA_edge_cbar = plt.colorbar(plt.cm.ScalarMappable(norm=MutA_edge_norm, cmap=MutA_edge_cmap), ax=ax, shrink=0.25, pad=0.05, anchor=(-1.25, -0.000001))
MutA_edge_cbar.set_label('Edges')

# Colorbar for node color mapping
MutA_cbar = plt.colorbar(plt.cm.ScalarMappable(norm=MutA_norm, cmap=MutA_cmap), ax=ax, shrink=0.25, pad=0.05, anchor=(0.0, 1.0))
MutA_cbar.set_label('Nodes')

plt.tight_layout()

plt.savefig('MutA_network_map.jpg', dpi=300)

# Show the plot
plt.show()


In [ ]:
# Filter the DataFrame
PC3_wgtd_MutA_pre_res_df = PC3_wgtd_MutA_pre_res.res2d[PC3_wgtd_MutA_pre_res.res2d['FDR q-val'] < 0.05]
PC3_wgtd_MutA_pre_res_df.head(100000)

In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_wgtd_MutA_all_genes = []

# Get the number of rows in the DataFrame
num_rows = len(PC3_wgtd_MutA_pre_res_df['Lead_genes'])

# Iterate over the first 10 rows or the available rows if less than 10
for i in range(min(1000, num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    lead_genes_value = PC3_wgtd_MutA_pre_res_df['Lead_genes'].iloc[i]
    PC3_wgtd_MutA_genes = lead_genes_value.split(";")
    PC3_wgtd_MutA_all_genes.extend(PC3_wgtd_MutA_genes)

# Remove duplicates from the list of genes
PC3_wgtd_MutA_unique_genes = list(set(PC3_wgtd_MutA_all_genes))

# Create a new DataFrame with unique genes
PC3_wgtd_MutA_unique_genes_df = pd.DataFrame(PC3_wgtd_MutA_unique_genes, columns=['Genes'])
PC3_wgtd_MutA_unique_genes_df.to_csv('PC3_wgtd_MutA_unique_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_wgtd_MutA_unique_genes_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gseapy as gp

# Assuming PC3_MutB_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
PC3_wgtd_MutB_rnk = PC3_stat_res3_rnk.copy()

# Sort the genes by rank in descending order
PC3_wgtd_MutB_rnk = PC3_wgtd_MutB_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
PC3_wgtd_MutB_ranking = PC3_wgtd_MutB_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
PC3_wgtd_MutB_ranking.to_csv('PC3_wgtd_MutB_ranking.csv', index=False)

# Perform gene set enrichment analysis
PC3_wgtd_MutB_pre_res = gp.prerank(rnk=PC3_wgtd_MutB_ranking, gene_sets='GO_Biological_Process_2023',
                                      seed=6, permutation_num=100, min_size=20, max_size=1000)

# Filter out terms with FDR q-value equal to zero
PC3_wgtd_MutB_pre_res.res2d = PC3_wgtd_MutB_pre_res.res2d[PC3_wgtd_MutB_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
PC3_wgtd_MutB_pre_res.res2d = PC3_wgtd_MutB_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
PC3_wgtd_MutB_terms = PC3_wgtd_MutB_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
PC3_wgtd_MutB_axs = PC3_wgtd_MutB_pre_res.plot(terms=PC3_wgtd_MutB_terms,
                                                     show_ranking=True,
                                                     figsize=(3, 4))

# Save the plot
plt.savefig('PC3_wgtd_MutB_rank.jpg', bbox_inches="tight")

PC3_wgtd_MutB_pre_res.res2d.to_csv('PC3_wgtd_MutB_pre_res.csv')
PC3_wgtd_MutB_pre_res.res2d


In [ ]:
# Filter the DataFrame
PC3_wgtd_MutB_pre_res_df = PC3_wgtd_MutB_pre_res.res2d[PC3_wgtd_MutB_pre_res.res2d['FDR q-val'] < 0.05]
PC3_wgtd_MutB_pre_res_df.head(100000)

In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_wgtd_MutB_all_genes = []

# Get the number of rows in the DataFrame
num_rows = len(PC3_wgtd_MutB_pre_res_df['Lead_genes'])

# Iterate over the first 1000 rows or the available rows if less than 1000
for i in range(min(1000, num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    lead_genes_value = PC3_wgtd_MutB_pre_res_df['Lead_genes'].iloc[i]
    PC3_wgtd_MutB_genes = lead_genes_value.split(";")
    PC3_wgtd_MutB_all_genes.extend(PC3_wgtd_MutB_genes)

# Remove duplicates from the list of genes
PC3_wgtd_MutB_unique_genes = list(set(PC3_wgtd_MutB_all_genes))

# Create a new DataFrame with unique genes
PC3_wgtd_MutB_unique_genes_df = pd.DataFrame(PC3_wgtd_MutB_unique_genes, columns=['Genes'])
PC3_wgtd_MutB_unique_genes_df.to_csv('PC3_wgtd_MutB_unique_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC3_wgtd_MutB_unique_genes_df)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_Control_unique_genes_df['Genes'])
genes_df2 = set(PC3_Control_sigs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_Control_unique_genes', 'PC3_Control_sigs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_Control_unique_genes_df['Genes'])
genes_df2 = set(PC3_Control_only_up_genes_df['Genes'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_Control_unique_genes', 'PC3_Control_only_up_genes_df'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_Control_unique_genes_df['Genes'])
genes_df2 = set(PC3_Control_only_down_genes_df['Genes'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_Control_unique_genes', 'PC3_Control_only_down_genes_df'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_Control_unique_genes_df['Genes'])
genes_df2 = set(PC3_wgtd_Genes_control_upDEGs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_Control_unique_genes', 'PC3_wgtd_Genes_control_upDEGs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_Control_unique_genes_df['Genes'])
genes_df2 = set(PC3_wgtd_Genes_control_downDEGs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_Control_unique_genes', 'PC3_wgtd_Genes_control_downDEGs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_MutA_unique_genes_df['Genes'])
genes_df2 = set(PC3_MutA_sigs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_MutA_unique_genes', 'PC3_MutA_sigs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_MutA_unique_genes_df['Genes'])
genes_df2 = set(PC3_wgtd_Genes_mutA_upDEGs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_MutA_unique_genes_df', 'PC3_wgtd_Genes_mutA_upDEGs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_MutA_unique_genes_df['Genes'])
genes_df2 = set(PC3_wgtd_Genes_mutA_downDEGs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_MutA_unique_genes_df', 'PC3_wgtd_Genes_mutA_downDEGs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_MutB_unique_genes_df['Genes'])
genes_df2 = set(PC3_MutB_sigs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_MutB_unique_genes_df', 'PC3_MutB_sigs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC3_wgtd_MutB_unique_genes_df['Genes'])
genes_df2 = set(PC3_wgtd_Genes_mutB_upDEGs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC3_wgtd_MutB_unique_genes_df', 'PC3_wgtd_Genes_mutB_upDEGs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes

In [ ]:
Normalised_dataset.head(2000000)

In [ ]:
pc2_sorted_df.head(10000000)

In [ ]:
Treated_PC2=pc2_sorted_df.abs()
Treated_PC2.head(100000)

In [ ]:
Norm_data_PC2_filtered_rearr = Normalised_dataset.reindex(Treated_PC2.index)
Norm_data_PC2_filtered_rearr

In [ ]:
PC2_weighted_Norm_dataset = Norm_data_PC2_filtered_rearr.mul(Treated_PC2.iloc[:, 0], axis=0)
PC2_weighted_Norm_dataset.to_csv('PC2_weighted_Norm_dataset.csv')
PC2_weighted_Norm_dataset.head(1000000)

In [ ]:
PC2_weighted_Norm_dataset1 = PC2_weighted_Norm_dataset.astype(int)
PC2_weighted_Norm_dataset1=PC2_weighted_Norm_dataset1.T
PC2_weighted_Norm_dataset1.head(1000000)

In [ ]:
metadata2 = pd.DataFrame(zip(PC2_weighted_Norm_dataset1.index, ['Control-Vehicle','Control-Vehicle','Control-Vehicle','Control-Treated','Control-Treated','Control-Treated','MutA-Vehicle', 'MutA-Vehicle', 'MutA-Vehicle','MutA-Treated', 'MutA-Treated', 'MutA-Treated', 'MutB-Vehicle', 'MutB-Vehicle', 'MutB-Vehicle','MutB-Treated', 'MutB-Treated', 'MutB-Treated']),
                        columns = ['Sample', 'Condition'])
metadata2 = metadata2.set_index('Sample')
metadata2

In [ ]:
dds2 = DeseqDataSet(counts=PC2_weighted_Norm_dataset1,
            metadata=metadata2,
            design_factors=["Condition"])

In [ ]:
dds2.deseq2()

In [ ]:
dds2

In [ ]:
PC2_stat_res1 = DeseqStats(dds2, contrast = ('Condition','MutA-Vehicle','MutB-Vehicle'))
PC2_stat_res1.summary()
PC2_stat_res1=PC2_stat_res1.results_df
PC2_stat_res1=pd.merge(PC2_stat_res1, data2[['gene_name']], left_index=True, right_index=True)
PC2_stat_res1

In [ ]:
# Copy the downregulated genes data
PC2_stat_res1_rnk = PC2_stat_res1.copy()

# Calculate the rank based on adjusted p-value
PC2_stat_res1_rnk['Rank'] = PC2_stat_res1.stat

# Sort the genes by rank in descending order
PC2_stat_res1_rnk = PC2_stat_res1_rnk.sort_values('Rank', ascending=False)
PC2_stat_res1_rnk

In [ ]:
PC2_res1 = PC2_stat_res1_rnk
PC2_res1

In [ ]:
PC2_MutA_vs_MutB_Vehicle_sigs = PC2_res1[(PC2_res1.padj < 0.05)]
PC2_MutA_vs_MutB_Vehicle_sigs.to_csv('PC2_MutA_vs_MutB_Vehicle_sigs.csv', index=True)
PC2_MutA_vs_MutB_Vehicle_sigs

In [ ]:
# Create MA plot for the first dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC2_res1['baseMean']), PC2_res1['log2FoldChange'], alpha=0.5, label='Total Genes', color='Gray')

# Create MA plot for the second dataset using log-transformed normalized counts for x-axis
plt.scatter(np.log10(PC2_MutA_vs_MutB_Vehicle_sigs['baseMean']), PC2_MutA_vs_MutB_Vehicle_sigs['log2FoldChange'], alpha=0.5, label='DEGs', color='blue')

# Add a horizontal line at y=0 for both datasets
plt.axhline(y=0, color='gray', linestyle='--')

# Set plot labels and title
plt.title('MA Plot - PC2 weighted MutA vs MutB Vehicle')
plt.xlabel('Log10 Normalized Counts')
plt.ylabel('Log2 Fold Change')

# Add legend
plt.legend()

plt.savefig('MA Plot - PC2 weighted MutA vs MutB Vehicle Total Genes & DEGs.jpg')

# Show the plot
plt.show()

In [ ]:
# MutA vs MutB Vehicle
PC2_MutA_vs_MutB_Vehicle_upregulated_genes_mask = PC2_MutA_vs_MutB_Vehicle_sigs['stat'] > 0
PC2_MutA_vs_MutB_Vehicle_downregulated_genes_mask = PC2_MutA_vs_MutB_Vehicle_sigs['stat'] < 0

PC2_MutA_vs_MutB_Vehicle_upregulated_genes = PC2_MutA_vs_MutB_Vehicle_sigs.loc[PC2_MutA_vs_MutB_Vehicle_upregulated_genes_mask]
PC2_MutA_vs_MutB_Vehicle_downregulated_genes = PC2_MutA_vs_MutB_Vehicle_sigs.loc[PC2_MutA_vs_MutB_Vehicle_downregulated_genes_mask]

print("Number of Upregulated Genes - MutA v/s MutB Vehicle:", len(PC2_MutA_vs_MutB_Vehicle_upregulated_genes))
print("\nMutA v/s MutB Vehicle Upregulated Genes:")
print(PC2_MutA_vs_MutB_Vehicle_upregulated_genes[['Rank', 'stat', 'padj', 'gene_name']])

PC2_MutA_vs_MutB_Vehicle_upregulated_genes.to_csv('PC2_MutA_vs_MutB_Vehicle_upregulated_genes.csv', index=True)

print("\n\nNumber of Downregulated Genes - MutA v/s MutB Vehicle :", len(PC2_MutA_vs_MutB_Vehicle_downregulated_genes))
print("\nMutA v/s MutB Vehicle  Downregulated Genes:")
print(PC2_MutA_vs_MutB_Vehicle_downregulated_genes[['Rank',  'stat', 'padj', 'gene_name']])

PC2_MutA_vs_MutB_Vehicle_downregulated_genes.to_csv('PC2_MutA_vs_MutB_Vehicle_downregulated_genes.csv', index=True)

In [ ]:
import matplotlib.pyplot as plt
from gseapy.plot import barplot
import gseapy as gp

# Enrichr analysis for GO Biological Processes (GOBP) - Upregulated Genes
PC2_CarvsMutB_enr_GOBP_up1 = gp.enrichr(
    gene_list=PC2_MutA_vs_MutB_Vehicle_upregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC2_CarvsMutB_enr_GOBP_up',
    cutoff=0.05
)

# Enrichr analysis for GO Biological Processes (GOBP) - Downregulated Genes
PC2_CarvsMutB_enr_GOBP_down1 = gp.enrichr(
    gene_list=PC2_MutA_vs_MutB_Vehicle_downregulated_genes['gene_name'],
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir='PC2_CarvsMutB_enr_GOBP_down',
    cutoff=0.05
)

# Plotting

# Bar plot for GO Biological Processes (Upregulated Genes)
barplot(PC2_CarvsMutB_enr_GOBP_up1.results, title="PC2 CarvsMutB: Enriched GOBP (Upregulated Genes)", cutoff=0.05, color='red')
plt.savefig('PC2_CarvsMutB_Enriched_GOBP_Upregulated_Genes.jpg', bbox_inches="tight")

# Bar plot for GO Biological Processes (Downregulated Genes)
barplot(PC2_CarvsMutB_enr_GOBP_down1.results, title="PC2 CarvsMutB: Enriched GOBP (Downregulated Genes)", cutoff=0.05, color='blue')
plt.savefig('PC2_CarvsMutB_Enriched_GOBP_Downregulated_Genes.jpg', bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gseapy as gp

# Assuming PC2_MutA_vs_MutB_upregulated_genes and 'GO_Biological_Process_2023' are defined somewhere in your code

# Copy the upregulated genes data
PC2_wgtd_MutA_vs_MutB_rnk = PC2_stat_res1_rnk.copy()

# Sort the genes by rank in descending order
PC2_wgtd_MutA_vs_MutB_rnk = PC2_wgtd_MutA_vs_MutB_rnk.sort_values('Rank', ascending=False).reset_index(drop=True)

# Extract gene names and their ranks
PC2_wgtd_MutA_vs_MutB_ranking = PC2_wgtd_MutA_vs_MutB_rnk[['gene_name', 'Rank']]

# Save gene names and ranks to a CSV file
PC2_wgtd_MutA_vs_MutB_ranking.to_csv('PC2_wgtd_MutA_vs_MutB_ranking.csv', index=False)

# Perform gene set enrichment analysis
PC2_wgtd_MutA_vs_MutB_pre_res = gp.prerank(rnk=PC2_wgtd_MutA_vs_MutB_ranking, gene_sets='GO_Biological_Process_2023',
                                      seed=6, permutation_num=100, min_size=20, max_size=1000)

# Filter out terms with FDR q-value equal to zero
PC2_wgtd_MutA_vs_MutB_pre_res.res2d = PC2_wgtd_MutA_vs_MutB_pre_res.res2d[PC2_wgtd_MutA_vs_MutB_pre_res.res2d['FDR q-val'] > 0]

# Sort the results by NES in descending order
PC2_wgtd_MutA_vs_MutB_pre_res.res2d = PC2_wgtd_MutA_vs_MutB_pre_res.res2d.sort_values('FDR q-val', ascending=True)

# Select the top 10 terms based on NES
PC2_wgtd_MutA_vs_MutB_terms = PC2_wgtd_MutA_vs_MutB_pre_res.res2d['Term'].head(10)

# Plot the top 10 terms based on NES
PC2_wgtd_MutA_vs_MutB_axs = PC2_wgtd_MutA_vs_MutB_pre_res.plot(terms=PC2_wgtd_MutA_vs_MutB_terms,
                                                     show_ranking=True,
                                                     figsize=(3, 4))

# Save the plot
plt.savefig('PC2_wgtd_MutA_vs_MutB_rank.jpg', bbox_inches="tight")

PC2_wgtd_MutA_vs_MutB_pre_res.res2d.to_csv('PC2_wgtd_MutA_vs_MutB_pre_res.csv')
PC2_wgtd_MutA_vs_MutB_pre_res.res2d

In [ ]:
# Filter the DataFrame
PC2_wgtd_MutA_vs_MutB_pre_res_df = PC2_wgtd_MutA_vs_MutB_pre_res.res2d[PC2_wgtd_MutA_vs_MutB_pre_res.res2d['FDR q-val'] < 0.05]
PC2_wgtd_MutA_vs_MutB_pre_res_df.head(100000)

In [ ]:
import pandas as pd

# Initialize an empty list to store genes
PC3_wgtd_MutA_vs_MutB_all_genes = []

# Get the number of rows in the DataFrame
num_rows = len(PC2_wgtd_MutA_vs_MutB_pre_res_df['Lead_genes'])

# Iterate over the first 1000 rows or the available rows if less than 1000
for i in range(min(1000, num_rows)):
    # Split the 'Lead_genes' column by ';' delimiter and add genes to the list
    lead_genes_value = PC2_wgtd_MutA_vs_MutB_pre_res_df['Lead_genes'].iloc[i]
    PC2_wgtd_MutA_vs_MutB_genes = lead_genes_value.split(";")
    PC2_wgtd_MutA_vs_MutB_genes.extend(PC2_wgtd_MutA_vs_MutB_genes)

# Remove duplicates from the list of genes
PC2_wgtd_MutA_vs_MutB_unique_genes = list(set(PC2_wgtd_MutA_vs_MutB_genes))

# Create a new DataFrame with unique genes
PC2_wgtd_MutA_vs_MutB_unique_genes_df = pd.DataFrame(PC2_wgtd_MutA_vs_MutB_unique_genes, columns=['Genes'])
PC2_wgtd_MutA_vs_MutB_unique_genes_df.to_csv('PC2_wgtd_MutA_vs_MutB_unique_genes_df.csv')

# Display the DataFrame containing unique genes
print(PC2_wgtd_MutA_vs_MutB_unique_genes_df)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pandas as pd

# Assuming PC3_wgtd_Control_unique_genes_df and PC3_Control_upregulated_genes are your dataframes

# Assuming the 'genes' column contains the gene names in both dataframes
genes_df1 = set(PC2_wgtd_MutA_vs_MutB_unique_genes_df['Genes'])
genes_df2 = set(PC2_MutA_vs_MutB_Vehicle_sigs['gene_name'])

# Create the Venn diagram
plt.figure(figsize=(8, 6))
venn2([genes_df1, genes_df2], ('PC2_wgtd_MutA_vs_MutB_unique_genes_df', 'PC2_MutA_vs_MutB_Vehicle_sigs'))
plt.title("Venn Diagram of Genes")
plt.show()

common_genes = genes_df1.intersection(genes_df2)
common_genes_list = list(common_genes)
print(common_genes_list)